In [15]:
%pip install pyvista
%pip install pandas
%pip install scipy
%pip install scikit-learn
%pip install alphashape
%pip install alphashape shapely
%pip install ortools
%pip install shapely




[notice] A new release of pip is available: 24.3.1 -> 25.1.1
[notice] To update, run: C:\Users\A2177802\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip



Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 25.1.1
[notice] To update, run: C:\Users\A2177802\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 25.1.1
[notice] To update, run: C:\Users\A2177802\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


...


[notice] A new release of pip is available: 24.3.1 -> 25.1.1
[notice] To update, run: C:\Users\A2177802\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 25.1.1
[notice] To update, run: C:\Users\A2177802\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 25.1.1
[notice] To update, run: C:\Users\A2177802\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [26]:
#Mitsumori senpai's code
import pyvista as pv
import csv 
import pandas as pd 
import numpy as np

mesh = pv.read('test_piece_3.stl')  # Load the STL file and name it "mesh"
points = mesh.points  # Get the point cloud from "mesh"
with open('dots.csv', 'w') as f:  # Create a new CSV file
    writer = csv.writer(f)  # Set to write mode
    writer.writerows(points)  # Write the point cloud

df = pd.read_csv('dots.csv')  # Open the CSV file with pandas
df = df.dropna()  # Remove blank rows
df = df.set_axis(['x', 'y', 'z'], axis='columns')  # Label columns as x, y, z (with space for row numbers)
maxi = df['z'].max() 
mini = df['z'].min() 
df = df[df['z'] > mini + 0.01]  # Remove the bottom plane
df.to_csv('dots.csv')  # Overwrite the CSV file
df = pd.read_csv('dots.csv')  # Re-open the CSV file with pandas
ll = [0, 0, 0]  # Specify that this is a 3-dimensional numeric array
pp = [0, 0, 200]  # Set a hypothetical initial tool position

while len(df) > 0:
    s = df['z'].max()  # Find the maximum value along the z-axis
    df_z = df[df['z'] == s]
    df_z.to_csv('layer.csv')
    df_z = pd.read_csv('layer.csv')
    df_z = df_z.drop(df_z.columns[[0, 1]], axis=1)
    df_z.to_csv('layer.csv')
    p = df_z['z'].idxmax()  # Maximum value on the z-axis, starting from this point for processing
    l = np.loadtxt('layer.csv', delimiter=',', dtype='float64', skiprows=1, usecols=[1, 2, 3])  # Convert from CSV to ndarray
    n = len(df_z)  # Get total number in df
    c = 0 
    if n == 1:
        pp = l
    ltp = [0, 0, 0]  # Tool path design order
    if n > 1:
        while n > 0:
            j = 0 
            ld = [] 
            if c == 3:
                l = np.vstack([l, llk]) 
                n = n + 1
            while j < n:
                ld.append(np.linalg.norm(pp - l[j]))  # Calculate and list distances from p
                j += 1
            k = ld.index(min(ld))  # Index of the smallest value in ld
            lk = l[k]  # Array at the index of the smallest value in l
            ltp = np.vstack([ltp, lk])  # Add lk vertically as an array to ltp
            if c == 0:
                llk = lk 
            ld.pop(k)  # Delete the k-th element in ld
            l = np.delete(l, k, 0)  # Delete the k-th row in l
            n -= 1  # Reduce n by one
            pp = lk  # Set next p as the current k
            c += 1
        ltp = np.delete(ltp, 0, 0) 
        ll = np.vstack([ll, ltp])
    else:
        ltp = np.delete(ltp, 0, 0) 
        ll = np.vstack([ll, l]) #Transform df to remaining layers
    df=df[df['z']<s]
ll = np.round(ll, 3)  # Round to 3 significant digits

with open('toolpath.csv', 'w') as f: 
    writer = csv.writer(f) 
    writer.writerows(ll)  # Save ll to CSV

df_ltp = pd.read_csv('toolpath.csv') 
df_ltp = df_ltp.set_axis(['x', 'y', 'z'], axis='columns') 
df_ltp.to_csv('toolpath.csv')  # Convert to G-code from here

l = np.loadtxt('toolpath.csv', delimiter=',', dtype='float64', skiprows=1, usecols=[1, 2, 3]) 
lx = l[:, 0]  # 0th column of l
ly = l[:, 1]  # 1st column of l
lz = l[:, 2]  # 2nd column of l

map1 = map(str, lx) 
lx = list(map1)  # Change data type to string
map2 = map(str, ly) 
ly = list(map2)  # Change data type to string
map3 = map(str, lz) 
lz = list(map3)  # Change data type to string

n = len(df_ltp) 
for i in range(n): 
    lx[i] = ('X' + lx[i])  # Add "X" prefix to n coordinates
for i in range(n): 
    ly[i] = ('Y' + ly[i])  # Add "Y" prefix to n coordinates
for i in range(n): 
    lz[i] = ('Z' + lz[i])  # Add "Z" prefix to n coordinates

df_gchord = pd.DataFrame(data={'G': 'G01', 'x': lx, 'y': ly, 'z': lz, 'EOB': ';'})
glist = df_gchord.values

outf = open("Gchord.csv", 'w', newline='') 
writer = csv.writer(outf) 
writer.writerows(glist) 
outf.close() 
df_g = pd.read_csv('Gchord.csv') 

df_g.to_csv('Gchord.txt', index=False, sep=" ")

In [1]:
# My code messing around with efficency
import pyvista as pv
import csv 
import pandas as pd 
import numpy as np

mesh = pv.read('freeform_test_7.stl')  # Load the STL file and name it "mesh"
points = mesh.points  # Get the point cloud from "mesh"
with open('dots.csv', 'w') as f:  # Create a new CSV file
    writer = csv.writer(f)  # Set to write mode
    writer.writerows(points)  # Write the point cloud

df = pd.read_csv('dots.csv')  # Open the CSV file with pandas
df = df.dropna()  # Remove blank rows
df = df.set_axis(['x', 'y', 'z'], axis='columns')  # Label columns as x, y, z (with space for row numbers)
maxi = df['z'].max() 
mini = df['z'].min() 
df = df[df['z'] > mini + 0.01]  # Remove the bottom plane
df.to_csv('dots.csv')  # Overwrite the CSV file
df = pd.read_csv('dots.csv')  # Re-open the CSV file with pandas
ll = [0, 0, 0]  # Specify that this is a 3-dimensional numeric array
pp = [0, 0, 200]  # Set a hypothetical initial tool position

while len(df) > 0:
    s = df['z'].max()  # Find the maximum value along the z-axis
    df_z = df[df['z'] == s]
    df_z.to_csv('layer.csv')
    df_z = pd.read_csv('layer.csv')
    df_z = df_z.drop(df_z.columns[[0, 1]], axis=1)
    df_z.to_csv('layer.csv')
    p = df_z['z'].idxmax()  # Maximum value on the z-axis, starting from this point for processing
    l = np.loadtxt('layer.csv', delimiter=',', dtype='float64', skiprows=1, usecols=[1, 2, 3])  # Convert from CSV to ndarray
    n = len(df_z)  # Get total number in df
    c = 0 
    if n == 1:
        pp = l
    ltp = [0, 0, 0]  # Tool path design order
    if n > 1:
        while n > 0:
            j = 0 
            ld = [] 
            if c == 3:
                l = np.vstack([l, llk]) 
                n = n + 1
            while j < n:
                ld.append(np.linalg.norm(pp - l[j]))  # Calculate and list distances from p
                j += 1
            k = ld.index(min(ld))  # Index of the smallest value in ld
            lk = l[k]  # Array at the index of the smallest value in l
            ltp = np.vstack([ltp, lk])  # Add lk vertically as an array to ltp
            if c == 0:
                llk = lk 
            ld.pop(k)  # Delete the k-th element in ld
            l = np.delete(l, k, 0)  # Delete the k-th row in l
            n -= 1  # Reduce n by one
            pp = lk  # Set next p as the current k
            c += 1
        ltp = np.delete(ltp, 0, 0) 
        ll = np.vstack([ll, ltp])
    else:
        ltp = np.delete(ltp, 0, 0) 
        ll = np.vstack([ll, l]) #Transform df to remaining layers
    df=df[df['z']<s]
ll = np.round(ll, 3)  # Round to 3 significant digits

with open('toolpath.csv', 'w') as f: 
    writer = csv.writer(f) 
    writer.writerows(ll)  # Save ll to CSV




toolpath = pd.read_csv('toolpath.csv', names=['x', 'y', 'z'], skiprows=1) # Load the tool path data from the CSV file


def is_layer_circular(points, tolerance=0.1):# Define a function to test circularity of a layer
    
    centroid = points[['x', 'y']].mean().values # Calculate the centroid of the layer in the x-y plane
    
    # Calculate distances of each point from the centroid
    distances = np.linalg.norm(points[['x', 'y']].values - centroid, axis=1)
    mean_distance = distances.mean()
    std_dev = distances.std()
    
    return (std_dev / mean_distance) < tolerance# Check if the relative standard deviation is within the tolerance


z_levels = toolpath['z'].unique() # Group tool path by z-level
circular_layers = 0

# Check circularity for each layer
for z in z_levels:
    layer_points = toolpath[toolpath['z'] == z]
    if len(layer_points) < 3:
        continue  # Skip layers with fewer than 3 points (not enough data to determine circularity)
    
    if is_layer_circular(layer_points):
        circular_layers += 1
        print(f"Layer at z={z} is circular.")
    else:
        print(f"Layer at z={z} is not circular.")

# Calculate the percentage of circular layers
total_layers = len(z_levels)
circularity_percentage = (circular_layers / total_layers) * 100
print(f"Circularity of tool path: {circularity_percentage:.2f}%")

# Decision based on threshold for circular layers
if circularity_percentage > 90:  # Arbitrary threshold, e.g., 90%
    while len(df) > 0:
        s = df['z'].max()  # Find the maximum value along the z-axis
    df_z = df[df['z'] == s]
    df_z.to_csv('layer.csv')
    df_z = pd.read_csv('layer.csv')
    df_z = df_z.drop(df_z.columns[[0, 1]], axis=1)
    df_z.to_csv('layer.csv')
    l = np.loadtxt('layer.csv', delimiter=',', dtype='float64', skiprows=1, usecols=[1, 2, 3])  # Convert from CSV to ndarray
    n = len(df_z)  # Get total number in df
    c = 0 
    if n == 1:
        pp = l
    ltp = [0, 0, 0]  # Tool path design order
    if n > 1:
        while n > 0:
            j = 0 
            ld = [] 
            if c == 3:
                l = np.vstack([l, llk]) 
                n = n + 1
            while j < n:
                ld.append(np.linalg.norm(pp - l[j]))  # Calculate and list distances from p
                j += 1
            k = ld.index(min(ld))  # Index of the smallest value in ld
            lk = l[k]  # Array at the index of the smallest value in l
            ltp = np.vstack([ltp, lk])  # Add lk vertically as an array to ltp
            if c == 0:
                llk = lk 
            ld.pop(k)  # Delete the k-th element in ld
            l = np.delete(l, k, 0)  # Delete the k-th row in l
            n -= 1  # Reduce n by one
            pp = lk  # Set next p as the current k
            c += 1
        ltp = np.delete(ltp, 0, 0) 
        ll = np.vstack([ll, ltp])
    else:
        ltp = np.delete(ltp, 0, 0) 
        ll = np.vstack([ll, l]) #Transform df to remaining layers
    df=df[df['z']<s]
    ll = np.round(ll, 3)  # Round to 3 significant digits
else:
    while len(df) > 0:
        # Select the current highest layer
        s = df['z'].max()
        df_z = df[df['z'] == s]
        
        # Sort points in the layer by both x and y coordinates
        df_z = df_z.sort_values(by=['y', 'x'])
        l = df_z[['x', 'y', 'z']].to_numpy()
        n = len(df_z)
        
        # Create square pattern toolpath
        if n > 1:
            layer_path = []
            while len(l) > 0:
                # Find min and max of x and y in the current subset
                min_x, max_x = l[:, 0].min(), l[:, 0].max()#
                min_y, max_y = l[:, 1].min(), l[:, 1].max()
                
                # Define the four sides of the current "square" ring
                top = l[(l[:, 1] == min_y) & (l[:, 0] >= min_x) & (l[:, 0] <= max_x)]
                right = l[(l[:, 0] == max_x) & (l[:, 1] > min_y) & (l[:, 1] <= max_y)]
                bottom = l[(l[:, 1] == max_y) & (l[:, 0] <= max_x) & (l[:, 0] >= min_x)][::-1]
                left = l[(l[:, 0] == min_x) & (l[:, 1] <= max_y) & (l[:, 1] > min_y)][::-1]
                
                # Combine the four sides to form a square ring path
                square_ring = np.vstack([top, right, bottom, left])
                layer_path.append(square_ring)
                
                # Remove the points in the current square ring from l
                l = np.array([point for point in l if point not in square_ring])
            
            # Add each square ring to the final toolpath (ll)
            for ring in layer_path:
                ll = np.vstack([ll, ring])

    # Update the remaining points by excluding the current layer
    df = df[df['z'] < s]

# Round the final tool path for cleaner G-code output
ll = np.round(ll, 3)

with open('toolpath.csv', 'w') as f:
    writer = csv.writer(f)
    writer.writerows(ll)



df_ltp = pd.read_csv('toolpath.csv') 
df_ltp = df_ltp.set_axis(['x', 'y', 'z'], axis='columns') 
df_ltp.to_csv('toolpath.csv')  # Convert to G-code from here

l = np.loadtxt('toolpath.csv', delimiter=',', dtype='float64', skiprows=1, usecols=[1, 2, 3]) 
lx = l[:, 0]  # 0th column of l
ly = l[:, 1]  # 1st column of l
lz = l[:, 2]  # 2nd column of l

map1 = map(str, lx) 
lx = list(map1)  # Change data type to string
map2 = map(str, ly) 
ly = list(map2)  # Change data type to string
map3 = map(str, lz) 
lz = list(map3)  # Change data type to string

n = len(df_ltp) 
for i in range(n): 
    lx[i] = ('X' + lx[i])  # Add "X" prefix to n coordinates
for i in range(n): 
    ly[i] = ('Y' + ly[i])  # Add "Y" prefix to n coordinates
for i in range(n): 
    lz[i] = ('Z' + lz[i])  # Add "Z" prefix to n coordinates

df_gchord = pd.DataFrame(data={'G': 'G01', 'x': lx, 'y': ly, 'z': lz, 'EOB': ';'})
glist = df_gchord.values

outf = open("Gchord.csv", 'w', newline='') 
writer = csv.writer(outf) 
writer.writerows(glist) 
outf.close() 
df_g = pd.read_csv('Gchord.csv') 

df_g.to_csv('Gchord.txt', index=False, sep=" ")


Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x0000026DE295F690>>
Traceback (most recent call last):
  File "C:\Users\A2177802\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\ipykernel\ipkernel.py", line 775, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(

KeyboardInterrupt: 


FileNotFoundError: File (C:\Users\A2177802\OneDrive - Sophia Univ. Students\Graduate Research\Python code\freeform_test_7.stl) not found

In [10]:
#11/12 code by end of the day (works not what looking for)

import pyvista as pv
import csv 
import pandas as pd 
import numpy as np

# Shift all points to make coordinates positive
min_x, min_y, min_z = points.min(axis=0)  # Calculate minimum for each axis
points[:, 0] -= min_x  # Shift X to start from 0
points[:, 1] -= min_y  # Shift Y to start from 0
points[:, 2] -= min_z  # Shift Z to start from 0


mesh = pv.read('freeform_test_7.stl')  # Load the STL file and name it "mesh"
points = mesh.points  # Get the point cloud from "mesh"
with open('dots.csv', 'w') as f:  # Create a new CSV file
        writer = csv.writer(f)  # Set to write mode
        writer.writerows(points)  # Write the point cloud

df = pd.read_csv('dots.csv')  # Open the CSV file with pandas
df = df.dropna()  # Remove blank rows
df = df.set_axis(['x', 'y', 'z'], axis='columns')  # Label columns as x, y, z (with space for row numbers)
maxi = df['z'].max() 
mini = df['z'].min() 
df = df[df['z'] > mini + 0.01]  # Remove the bottom plane
df.to_csv('dots.csv')  # Overwrite the CSV file
df = pd.read_csv('dots.csv')  # Re-open the CSV file with pandas
ll = [0, 0, 0]  # Specify that this is a 3-dimensional numeric array
pp = [0, 0, 200]  # Set a hypothetical initial tool position

# Parameters
tolerance = 0.1  # Tolerance for grouping Z coordinates into contours

# Round all Z values in the DataFrame before starting the loop
df['z'] = df['z'].round(0)
df['x'] = df['x'].round(1)
df['y'] = df['y'].round(1)
df_z_sorted = df_z.sort_values(by=['x', 'y'])  # Sort by x and then y

while len(df) > 0:
    s = df['z'].max()  # Find the maximum Z value for the current layer
    
    # Select all points close to the current max Z within the tolerance
    df_z = df[(df['z'] >= s - tolerance) & (df['z'] <= s + tolerance)]
    df_x = df[(df['x'] >= s - tolerance) & (df['x'] <= s + tolerance)]
    df_y = df[(df['y'] >= s - tolerance) & (df['y'] <= s + tolerance)]
    df_z.to_csv('layer.csv')

    # Process layer as before
    df_z = pd.read_csv('layer.csv')
    df_z = df_z.drop(df_z.columns[[0, 1]], axis=1)
    df_x = df_x.drop(df_x.columns[[0, 1]], axis=1)
    df_y = df_y.drop(df_y.columns[[0, 1]], axis=1)
    df_z.to_csv('layer.csv')

    # Convert from CSV to ndarray for tool path generation
    l = np.loadtxt('layer.csv', delimiter=',', dtype='float64', skiprows=1, usecols=[1, 2, 3])
    n = len(df_z)  # Total points in this layer
    c = 0
    if n == 1:
        pp = l
    ltp = [0, 0, 0]

    # Tool path generation logic (unchanged)
    if n > 1:
        while n > 0:
            j = 0
            ld = []
            if c == 3:
                l = np.vstack([l, llk])
                n = n + 1
            while j < n:
                ld.append(np.linalg.norm(pp - l[j]))
                j += 1
            k = ld.index(min(ld))
            lk = l[k]
            ltp = np.vstack([ltp, lk])
            if c == 0:
                llk = lk
            ld.pop(k)
            l = np.delete(l, k, 0)
            n -= 1
            pp = lk
            c += 1
        ltp = np.delete(ltp, 0, 0)
        ll = np.vstack([ll, ltp])
    else:
        ltp = np.delete(ltp, 0, 0)
        ll = np.vstack([ll, l])

    # Remove processed points from df based on tolerance
    df = df[df['z'] < s - tolerance]

ll = np.round(ll, 3)  # Final rounding



with open('toolpath.csv', 'w') as f: 
        writer = csv.writer(f) 
        writer.writerows(ll)  # Save ll to CSV

df_ltp = pd.read_csv('toolpath.csv') 
df_ltp = df_ltp.set_axis(['x', 'y', 'z'], axis='columns') 



df_ltp.to_csv('toolpath.csv')  # Convert to G-code from here

l = np.loadtxt('toolpath.csv', delimiter=',', dtype='float64', skiprows=1, usecols=[1, 2, 3]) 
lx = l[:, 0]  # 0th column of l
ly = l[:, 1]  # 1st column of l
lz = l[:, 2]  # 2nd column of l

map1 = map(str, lx) 
lx = list(map1)  # Change data type to string
map2 = map(str, ly) 
ly = list(map2)  # Change data type to string
map3 = map(str, lz) 
lz = list(map3)  # Change data type to string

n = len(df_ltp) 
for i in range(n): 
    lx[i] = ('X' + lx[i])  # Add "X" prefix to n coordinates
for i in range(n): 
    ly[i] = ('Y' + ly[i])  # Add "Y" prefix to n coordinates
for i in range(n): 
    lz[i] = ('Z' + lz[i])  # Add "Z" prefix to n coordinates

df_gchord = pd.DataFrame(data={'G': 'G01', 'x': lx, 'y': ly, 'z': lz, 'EOB': ';'})

glist = df_gchord.values

# Mark duplicates
duplicates = df_gchord.duplicated(subset=['x', 'y', 'z'], keep=False)  # Mark all duplicates

# Mark the first and last occurrence of each duplicate group
first_occurrence = df_gchord.duplicated(subset=['x', 'y', 'z'], keep='first')  # Mark duplicates after the first occurrence
last_occurrence = df_gchord.duplicated(subset=['x', 'y', 'z'], keep='last')  # Mark duplicates before the last occurrence

# Combine the masks to keep only the first and last occurrence of duplicates
df_gchord_clean = df_gchord[~first_occurrence | ~last_occurrence]

# Convert the cleaned DataFrame to a numpy array for writing to CSV
glist = df_gchord_clean.values

outf = open("Gchord.csv", 'w', newline='') 
writer = csv.writer(outf) 
writer.writerows(glist) 
outf.close() 
df_g = pd.read_csv('Gchord.csv') 


df_g.to_csv('Gchord.txt', index=False, sep=" ")

In [11]:
#11/12 test area (works not what looking for)

import pyvista as pv
import csv 
import pandas as pd 
import numpy as np

# Load the STL file and extract points
mesh = pv.read('freeform_test_7.stl')  # Load the STL file and name it "mesh"
points = mesh.points  # Get the point cloud from "mesh"

# Write the points to a CSV file
with open('dots.csv', 'w', newline='') as f:
    writer = csv.writer(f)  # Set to write mode
    writer.writerows(points)  # Write the point cloud

# Load the points into a pandas DataFrame
df = pd.read_csv('dots.csv', header=None)  # Read the points without headers
df = df.dropna()  # Remove any rows with NaN values
df.columns = ['x', 'y', 'z']  # Assign column names for clarity

# Basic filtering
maxi = df['z'].max() 
mini = df['z'].min() 
df = df[df['z'] > mini + 0.01]  # Remove the bottom plane
df.to_csv('dots.csv', index=False)  # Overwrite the CSV file

# Parameters for tolerance and rounding
tolerance = 0.1  # Tolerance for grouping Z coordinates into contours
df['z'] = df['z'].round(0)  # Round Z values to the nearest integer
df['x'] = df['x'].round(1)  # Round X values to one decimal place
df['y'] = df['y'].round(1)  # Round Y values to one decimal place

ll = []  # List to hold the toolpath
pp = np.array([0, 0, 200])  # Set an initial tool position

# Process layers
while len(df) > 0:
    s = df['z'].max()  # Find the maximum Z value for the current layer
    
    # Select all points at the current Z level (within tolerance)
    df_z = df[(df['z'] >= s - tolerance) & (df['z'] <= s + tolerance)]
    df_z.to_csv('layer.csv', index=False)  # Save layer to CSV for further processing
    
    # Convert to NumPy array
    l = np.loadtxt('layer.csv', delimiter=',', dtype='float64', skiprows=1, usecols=[0, 1, 2])
    
    n = len(df_z)  # Number of points in the current layer
    c = 0  # Counter for toolpath points
    
    if n == 1:
        pp = l[0]  # If there's only one point, set it as the new tool position
        ll.append(pp)  # Add this point to the toolpath
    else:
        # Generate the toolpath by iterating through the points
        while n > 0:
            ld = [np.linalg.norm(pp - l[i]) for i in range(n)]  # Calculate distances to current tool position
            k = ld.index(min(ld))  # Find the index of the nearest point
            lk = l[k]  # Get the nearest point
            ll.append(lk)  # Add the point to the toolpath
            
            pp = lk  # Update the tool position
            l = np.delete(l, k, axis=0)  # Remove the chosen point from the list
            n -= 1  # Decrease the number of remaining points
    
    # After processing this layer, remove the points with this Z value from df
    df = df[df['z'] < s - tolerance]

# Convert the final toolpath into a NumPy array and round the values
ll = np.round(ll, 3)

# Save the toolpath to a CSV file
with open('toolpath.csv', 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerows(ll)  # Write toolpath to CSV

# Load the toolpath and prepare for G-code generation
df_ltp = pd.read_csv('toolpath.csv', header=None)
df_ltp.columns = ['x', 'y', 'z']  # Rename columns for clarity

# Convert to G-code format
lx = df_ltp['x'].astype(str).apply(lambda x: 'X' + x).tolist()
ly = df_ltp['y'].astype(str).apply(lambda x: 'Y' + x).tolist()
lz = df_ltp['z'].astype(str).apply(lambda x: 'Z' + x).tolist()

# Create the G-code DataFrame
df_gchord = pd.DataFrame(data={'G': 'G01', 'x': lx, 'y': ly, 'z': lz, 'EOB': ';'})

# Remove duplicates, keeping only the first and last occurrence of each point (x, y, z)
duplicates_first_last = df_gchord.duplicated(subset=['x', 'y', 'z'], keep=False)  # Identify all duplicates
first_occurrence = df_gchord.duplicated(subset=['x', 'y', 'z'], keep='first')  # Mark after the first occurrence
last_occurrence = df_gchord.duplicated(subset=['x', 'y', 'z'], keep='last')  # Mark before the last occurrence

# Filter the DataFrame to keep only the first and last occurrence of duplicates
df_gchord_clean = df_gchord[~first_occurrence | ~last_occurrence]

# Write the cleaned toolpath to a CSV file
with open("Gchord.csv", 'w', newline='') as outf:
    writer = csv.writer(outf)
    writer.writerows(df_gchord_clean.values)  # Write the cleaned toolpath

# Save the cleaned toolpath as a text file
df_gchord_clean.to_csv('Gchord.txt', index=False, sep=" ")

print("Toolpath generated and saved successfully.")


Toolpath generated and saved successfully.


In [12]:
# 11/13 code by end of day (works not what looking for)

import pyvista as pv
import csv 
import pandas as pd 
import numpy as np
from scipy.spatial import ConvexHull  # Import ConvexHull


mesh = pv.read('freeform_test_7.stl')  # Load the STL file and name it "mesh"
points = mesh.points  # Get the point cloud from "mesh"

with open('dots.csv', 'w') as f:  # Create a new CSV file
        writer = csv.writer(f)  # Set to write mode
        writer.writerows(points)  # Write the point cloud

df = pd.read_csv('dots.csv')  # Open the CSV file with pandas
df = df.dropna()  # Remove blank rows
df = df.set_axis(['x', 'y', 'z'], axis='columns')  # Label columns as x, y, z

maxi = df['z'].max() 
mini = df['z'].min() 
df = df[df['z'] > mini + 0.1]  # Remove the bottom plane

df.to_csv('dots.csv')  # Overwrite the CSV file
df = pd.read_csv('dots.csv')  # Re-open the CSV file with pandas

ll = [0, 0, 0]  # Specify that this is a 3-dimensional numeric array
pp = [0, 0, 400]  # Set a hypothetical initial tool position


# Parameters
tolerance = 0.1  # Tolerance for grouping Z coordinates into contours

# # Round all Z values in the DataFrame before starting the loop
df['z'] = df['z'].round(0)
df['x'] = df['x'].round(0)
df['y'] = df['y'].round(0)

while len(df) > 0:
    s = df['z'].max()  # Find the maximum Z value for the current layer
    
    # Select all points close to the current max Z within the tolerance, generates the layers
    df_z = df[(df['z'] >= s - tolerance) & (df['z'] <= s + tolerance)]
    df_z.to_csv('layer.csv')

    # Skip the layer if it has fewer than 3 points
    if len(df_z) < 3:
        # Remove the processed points and continue to the next layer
        df = df[df['z'] < s - tolerance]
        continue

    # Calculate the convex hull for the layer, generates the "max" x and y points for the layer
    if len(df_z) > 2:  # ConvexHull requires at least 3 points
        points_layer = df_z[['x', 'y']].values  # Only use x, y coordinates
        
        hull = ConvexHull(points_layer)
        
        # Keep only the points that are on the convex hull
        df_z = df_z.iloc[hull.vertices]

    # Convert from CSV to ndarray for tool path generation
    l = np.array(df_z[['x', 'y', 'z']].values, dtype='float64')
    n = len(df_z)  # Total points in this layer
    c = 0
    
    if n == 1:
        pp = l
    ltp = [0, 0, 0]

    # Tool path generation logic (unchanged)
    if n > 1:
        while n > 0:
            j = 0 
            ld = [] 
            if c == 3:
                l = np.vstack([l, llk]) 
                n = n + 1
            while j < n:
                ld.append(np.sum(np.abs(pp - l[j])))  # Calculate and list distances from p
                j += 1
            k = ld.index(min(ld))  # Index of the smallest value in ld
            lk = l[k]  # Array at the index of the smallest value in l
            ltp = np.vstack([ltp, lk])  # Add lk vertically as an array to ltp
            if c == 0:
                llk = lk 
            ld.pop(k)  # Delete the k-th element in ld
            l = np.delete(l, k, 0)  # Delete the k-th row in l
            n -= 1  # Reduce n by one
            pp = lk  # Set next p as the current k
            c += 1
        ltp = np.delete(ltp, 0, 0) 
        ll = np.vstack([ll, ltp])

    # Remove processed points from df based on tolerance
    df = df[df['z'] < s - tolerance]

with open('toolpath.csv', 'w') as f: 
    writer = csv.writer(f) 
    writer.writerows(ll)  # Save ll to CSV

df_ltp = pd.read_csv('toolpath.csv') 
df_ltp = df_ltp.set_axis(['x', 'y', 'z'], axis='columns') 


df_ltp.to_csv('toolpath.csv')  # Convert to G-code from here

l = np.loadtxt('toolpath.csv', delimiter=',', dtype='float64', skiprows=1, usecols=[1, 2, 3]) 
lx = l[:, 0]  # 0th column of l
ly = l[:, 1]  # 1st column of l
lz = l[:, 2]  # 2nd column of l

map1 = map(str, lx) 
lx = list(map1)  # Change data type to string
map2 = map(str, ly) 
ly = list(map2)  # Change data type to string
map3 = map(str, lz) 
lz = list(map3)  # Change data type to string

n = len(df_ltp) 
for i in range(n): 
    lx[i] = ('X' + lx[i])  # Add "X" prefix to n coordinates
for i in range(n): 
    ly[i] = ('Y' + ly[i])  # Add "Y" prefix to n coordinates
for i in range(n): 
    lz[i] = ('Z' + lz[i])  # Add "Z" prefix to n coordinates

df_gchord = pd.DataFrame(data={'G': 'G01', 'x': lx, 'y': ly, 'z': lz, 'EOB': ';'})
glist = df_gchord.values

outf = open("Gchord.csv", 'w', newline='') 
writer = csv.writer(outf) 
writer.writerows(glist) 
outf.close() 
df_g = pd.read_csv('Gchord.csv') 


df_g.to_csv('Gchord.txt', index=False, sep=" ")

In [ ]:
# 11/14 code by the end of the day (does not work)

import pyvista as pv
import csv 
import pandas as pd 
import numpy as np
from scipy.spatial import ConvexHull  # Import ConvexHull
from scipy.spatial.distance import cdist

mesh = pv.read('freeform_test_7.stl')  # Load the STL file and name it "mesh"
points = mesh.points  # Get the point cloud from "mesh"

with open('dots.csv', 'w') as f:  # Create a new CSV file
        writer = csv.writer(f)  # Set to write mode
        writer.writerows(points)  # Write the point cloud

df = pd.read_csv('dots.csv')  # Open the CSV file with pandas
df = df.dropna()  # Remove blank rows
df = df.set_axis(['x', 'y', 'z'], axis='columns')  # Label columns as x, y, z

maxi = df['z'].max() 
mini = df['z'].min() 
df = df[df['z'] > mini + 0.1]  # Remove the bottom plane

df.to_csv('dots.csv')  # Overwrite the CSV file
df = pd.read_csv('dots.csv')  # Re-open the CSV file with pandas

ll = [0, 0, 0]  # Specify that this is a 3-dimensional numeric array
pp = [0, 0, 200]  # Set a hypothetical initial tool position


# Parameters
tolerance = 0.1  # Tolerance for grouping Z coordinates into contours

# # Round all Z values in the DataFrame before starting the loop
df['z'] = df['z'].round(0)
df['x'] = df['x'].round(0)
df['y'] = df['y'].round(0)

while len(df) > 0:
    s = df['z'].max()  # Find the maximum Z value for the current layer
    
    # Select all points close to the current max Z within the tolerance, generates the layers
    df_z = df[(df['z'] >= s - tolerance) & (df['z'] <= s + tolerance)]
    df_z.to_csv('layer.csv')

    # Skip the layer if it has fewer than 3 points
    if len(df_z) < 3:
        # Remove the processed points and continue to the next layer
        df = df[df['z'] < s - tolerance]
        continue

    # Calculate the convex hull for the layer, generates the "max" x and y points for the layer
    if len(df_z) > 2:  # ConvexHull requires at least 3 points
        points_layer = df_z[['x', 'y']].values  # Only use x, y coordinates
        
        hull = ConvexHull(points_layer)
        
        # Keep only the points that are on the convex hull
        df_z = df_z.iloc[hull.vertices]

    # Convert from CSV to ndarray for tool path generation
    l = np.array(df_z[['x', 'y', 'z']].values, dtype='float64')
    n = len(df_z)  # Total points in this layer
    c = 0

    initial_point = pp  # Keep track of the initial point
    visited = [False] * len(l)  # List of visited flags for each point

    if n == 1:
        pp = l
    ltp = [0, 0, 0]

    # Tool path generation logic
    if n > 1:
        visited[0] = True  # Mark the initial point as visited initially      
        # Initialize the direction vector with None
        previous_direction = None
        ltp = []  # Initialize layer tool path as an empty list  
        while n > 0:
            j = 0 
            ld = [] 
            
            if c == 3:
                l = np.vstack([l, llk]) 
                n = n + 1
            
            while j < n:
                # Skip the initial point until we've visited at all the points
                if np.array_equal(l[j], initial_point) and n == 0:
                    j += 1
                    continue  # Skip the initial point

                # Calculate distance from current position (pp) to the point (l[j])
                ld.append(np.sum(np.abs(pp - l[j])))  # Calculate and list distances from p
                j += 1
                
                for j in range(len(l)):
                    if not visited[j]:  # Only consider unvisited points
                        # Calculate the vector to the point
                        next_vector = l[j] - pp
                        distance = np.linalg.norm(next_vector)  # Euclidean distance

                        # Calculate the angle with the previous direction vector
                        if previous_direction is not None:
                            angle = np.arccos(
                                np.dot(previous_direction, next_vector) /
                                (np.linalg.norm(previous_direction) * np.linalg.norm(next_vector))
                            )
                        else:
                            angle = 0  # No previous direction, treat as zero angle

                        # Store distance, angle, and index
                        ld.append((distance, angle, j))

                if not ld:
                    ltp.append(initial_point)  # Return to initial point to complete the layer
                    break

                k = ld.index(min(ld))  # Index of the smallest value in ld
                lk = l[k]  # Array at the index of the smallest value in l
                ltp = np.vstack([ltp, lk])  # Add lk vertically as an array to ltp

                if c == 0:
                    llk = lk

                if n == 0:    
                    visited[0] = False  # Unmark the initial point as visited to allow revisiting
                    ltp = np.vstack([ltp, initial_point])  # Add initial point to complete the loop
                    l = np.vstack([l, llk]) 
                    n = n + 1
                    
    
                ld.pop(k)  # Delete the k-th element in ld
                l = np.delete(l, k, 0)  # Delete the k-th row in l
                n -= 1  # Reduce n by one
                
                pp = lk  # Set next p as the current k
                c += 1
        
        ltp = np.delete(ltp, 0, 0) 
        ll = np.vstack([ll, ltp])

    # Remove processed points from df based on tolerance
    df = df[df['z'] < s - tolerance]


with open('toolpath.csv', 'w') as f: 
    writer = csv.writer(f) 
    writer.writerows(ll)  # Save ll to CSV

df_ltp = pd.read_csv('toolpath.csv') 
df_ltp = df_ltp.set_axis(['x', 'y', 'z'], axis='columns') 


df_ltp.to_csv('toolpath.csv')  # Convert to G-code from here

l = np.loadtxt('toolpath.csv', delimiter=',', dtype='float64', skiprows=1, usecols=[1, 2, 3]) 
lx = l[:, 0]  # 0th column of l
ly = l[:, 1]  # 1st column of l
lz = l[:, 2]  # 2nd column of l

map1 = map(str, lx) 
lx = list(map1)  # Change data type to string
map2 = map(str, ly) 
ly = list(map2)  # Change data type to string
map3 = map(str, lz) 
lz = list(map3)  # Change data type to string

n = len(df_ltp) 
for i in range(n): 
    lx[i] = ('X' + lx[i])  # Add "X" prefix to n coordinates
for i in range(n): 
    ly[i] = ('Y' + ly[i])  # Add "Y" prefix to n coordinates
for i in range(n): 
    lz[i] = ('Z' + lz[i])  # Add "Z" prefix to n coordinates

df_gchord = pd.DataFrame(data={'G': 'G01', 'x': lx, 'y': ly, 'z': lz, 'EOB': ';'})
glist = df_gchord.values

outf = open("Gchord.csv", 'w', newline='') 
writer = csv.writer(outf) 
writer.writerows(glist) 
outf.close() 
df_g = pd.read_csv('Gchord.csv') 


df_g.to_csv('Gchord.txt', index=False, sep=" ")

ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()

In [ ]:
# 11/15 code by the end of the day, figure out how to get the tool path to go to the initial point after first processing all of the points on the layer (works not what looking for)
import pyvista as pv
import csv
import pandas as pd
import numpy as np
from scipy.spatial import ConvexHull

# Load mesh and prepare CSV file of points
mesh = pv.read('freeform_test_7.stl')
points = mesh.points
with open('dots.csv', 'w') as f:
    writer = csv.writer(f)
    writer.writerows(points)

df = pd.read_csv('dots.csv').dropna()
df = df.set_axis(['x', 'y', 'z'], axis='columns')

# Remove points close to the bottom plane
mini = df['z'].min()
df = df[df['z'] > mini + 0.1]
df.to_csv('dots.csv')

# Initial variables
ll = [0, 0, 0]
pp = [0, 0, 400]  # Initial tool position
tolerance = 0.1  # Z tolerance for grouping points into layers

# Round values for layer selection
df['z'] = df['z'].round(0)
df['x'] = df['x'].round(0)
df['y'] = df['y'].round(0)

while len(df) > 0:
    s = df['z'].max()
    df_z = df[(df['z'] >= s - tolerance) & (df['z'] <= s + tolerance)]
    df_z.to_csv('layer.csv')

    if len(df_z) < 3:
        df = df[df['z'] < s - tolerance]
        continue

    if len(df_z) > 2:
        points_layer = df_z[['x', 'y']].values
        hull = ConvexHull(points_layer)
        df_z = df_z.iloc[hull.vertices]

    # Store all points in this layer as an array
    l = np.array(df_z[['x', 'y', 'z']].values, dtype='float64')
    n = len(df_z)

    # Track the initial point (first point in the layer)
    initial_point = l[0] if n > 0 else None

    # Process all points except the initial one
    ltp = [0, 0, 0]
    if n > 1:
        processed_initial = False
        while n > 0:
            j = 0
            ld = []

            # Calculate distances, skip the initial point temporarily
            while j < n:
                if not np.array_equal(l[j], initial_point):
                    ld.append(np.sum(np.abs(pp - l[j])))
                else:
                    ld.append(float('inf'))  # Skip initial point temporarily
                j += 1

            # Find closest point and retrieve its coordinates
            k = ld.index(min(ld))
            lk = l[k]
            ltp = np.vstack([ltp, lk])

            # Update remaining points
            l = np.delete(l, k, 0)
            n -= 1
            pp = lk

        # Now add the initial point explicitly after all other points have been processed
        if initial_point is not None:
            ltp = np.vstack([ltp, initial_point])

        # Remove the initial empty row (the first row added to ltp)
        ltp = np.delete(ltp, 0, 0)
        ll = np.vstack([ll, ltp])

    # Remove processed points from df based on Z tolerance
    df = df[df['z'] < s - tolerance]

# Save the toolpath to a CSV
with open('toolpath.csv', 'w') as f:
    writer = csv.writer(f)
    writer.writerows(ll)

# Process for G-code conversion
df_ltp = pd.read_csv('toolpath.csv')
df_ltp = df_ltp.set_axis(['x', 'y', 'z'], axis='columns')
df_ltp.to_csv('toolpath.csv')

# Convert coordinates to G-code format
l = np.loadtxt('toolpath.csv', delimiter=',', dtype='float64', skiprows=1, usecols=[1, 2, 3])
lx, ly, lz = l[:, 0].astype(str), l[:, 1].astype(str), l[:, 2].astype(str)
lx = ['X' + x for x in lx]
ly = ['Y' + y for y in ly]
lz = ['Z' + z for z in lz]

# Write G-code commands to CSV
df_gchord = pd.DataFrame({'G': 'G01', 'x': lx, 'y': ly, 'z': lz, 'EOB': ';'})
with open("Gchord.csv", 'w', newline='') as outf:
    writer = csv.writer(outf)
    writer.writerows(df_gchord.values)

df_g = pd.read_csv('Gchord.csv')
df_g.to_csv('Gchord.txt', index=False, sep=" ")



In [8]:
# 11/18 code by the end of the day, figure out how to get the tool path to go to the initial point after first processing all of the points on the layer 
import pyvista as pv
import csv
import pandas as pd
import numpy as np
from scipy.spatial import ConvexHull

# Load mesh and prepare CSV file of points
mesh = pv.read('freeform_test_7.stl')
points = mesh.points
with open('dots.csv', 'w') as f:
    writer = csv.writer(f)
    writer.writerows(points)

df = pd.read_csv('dots.csv').dropna()
df = df.set_axis(['x', 'y', 'z'], axis='columns')

# Remove points close to the bottom plane
mini = df['z'].min()
df = df[df['z'] > mini + 0.1]
df.to_csv('dots.csv')

# Initial variables
ll = []  # Tool path storage
pp = [0, 0, 400]  # Initial tool position
tolerance = 0  # Z tolerance for grouping points into layers

# Round values for layer selection
df['z'] = df['z'].round(0)
df['x'] = df['x'].round(0)
df['y'] = df['y'].round(0)

while len(df) > 0:
    # Identify the highest layer
    s = df['z'].max()
    df_z = df[(df['z'] >= s - tolerance) & (df['z'] <= s + tolerance)]

    # if len(df_z) < 3: #checks if the layer has less than 3 points (convexhull requires at least 3 points)
    #     df = df[df['z'] < s - tolerance]
    #     continue

    if len(df_z) > 2:
        points_layer = df_z[['x', 'y']].values
        hull = ConvexHull(points_layer)
        df_z = df_z.iloc[hull.vertices]

    # Store all points in this layer as an array
    l = np.array(df_z[['x', 'y', 'z']].values, dtype='float64')
    n = len(df_z)

    # Process all points in the current layer
    initial_point = l[0]  # First point of the layer
    ltp = [pp, initial_point]  # Start the layer by visiting the initial point
    pp = initial_point  # Update the tool position to the initial point

    while n > 0:
        distances = np.linalg.norm(np.sum(np.abs(l - pp,)))
        #distances = np.sum(np.abs(l - pp))
        #distances = np.linalg.norm(l - pp, axis=1)
        #distances = np.linalg.norm(np.abs(l - pp))
        #distances = np.linalg.norm(np.sum(l - pp))

        closest_idx = np.argmin(distances)
        closest_point = l[closest_idx]
        
        # Add the closest point to the tool path
        ltp.append(closest_point)
        pp = closest_point

        # Remove the processed point from the list
        l = np.delete(l, closest_idx, axis=0)
        n -= 1

    ltp.append(initial_point)
    pp = initial_point  # Update the tool position to the initial point

    # Append the current layer's tool path to the full tool path
    ll.extend(ltp)

    # Remove processed points from df based on Z tolerance
    df = df[df['z'] < s - tolerance]
    
# Save the toolpath to a CSV
with open('toolpath.csv', 'w') as f:
    writer = csv.writer(f)
    writer.writerows(ll)

# Process for G-code conversion
df_ltp = pd.DataFrame(ll, columns=['x', 'y', 'z'])
df_ltp.to_csv('toolpath.csv', index=False)

# Convert coordinates to G-code format
l = np.array(ll)
lx, ly, lz = l[:, 0].astype(str), l[:, 1].astype(str), l[:, 2].astype(str)
lx = ['X' + x for x in lx]
ly = ['Y' + y for y in ly]
lz = ['Z' + z for z in lz]

# Write G-code commands to CSV
df_gchord = pd.DataFrame({'G': 'G01', 'x': lx, 'y': ly, 'z': lz, 'EOB': ';'})
with open("Gchord.csv", 'w', newline='') as outf:
    writer = csv.writer(outf)
    writer.writerows(df_gchord.values)

df_g = pd.read_csv('Gchord.csv')
df_g.to_csv('Gchord.txt', index=False, sep=" ")


In [ ]:
# 11/21 code by the end of the day, figure out how to get the program to do the TSP from the "closest_point_next_layer" so that there is no jumping from point to point across the work piece


import pyvista as pv
import csv
import pandas as pd
import numpy as np
from scipy.spatial import ConvexHull
pd.options.mode.copy_on_write = True

# Load mesh and prepare CSV file of points
mesh = pv.read('freeform_test_7.stl')
points = mesh.points
with open('dots.csv', 'w') as f:
    writer = csv.writer(f)
    writer.writerows(points)

df = pd.read_csv('dots.csv').dropna()
df = df.set_axis(['x', 'y', 'z'], axis='columns')

# Remove points close to the bottom plane
df = df[df['z'] > df['z'].min() + 0.1]

# Initial variables
ll = []  # Tool path storage
pp = [0, 0, 400]  # Initial tool position
tolerance = 0.1  # Z tolerance for grouping points into layers

# Round values for layer selection
df['z'] = df['z'].round(0)
df['x'] = df['x'].round(0)
df['y'] = df['y'].round(0)

# Process all points in the current layer
initial_point = pp  # First point of the layer
ltp = [pp, initial_point]  # Start the layer path


# Loop through layers
while len(df) > 0:
    # Identify the highest layer
    s = df['z'].max()
    df_z = df[(df['z'] >= s - tolerance) & (df['z'] <= s + tolerance)]

    # If the layer has less than 3 points, skip it
    if len(df_z) < 3:
        df = df[df['z'] < s - tolerance]
        continue

    # Filter points to retain only those near the boundary
    origin = np.array([0, 0, 0])
    distances = np.linalg.norm(df_z[['x', 'y', 'z']].values - origin, axis=1)
    df_z['distance'] = distances
    max_distance = df_z['distance'].max()
    df_z = df_z[df_z['distance'] >= max_distance * 0.9]

    # Apply ConvexHull to filtered points
    points_layer = df_z[['x', 'y']].values
    hull = ConvexHull(points_layer)
    df_z = df_z.iloc[hull.vertices]

    # Store all points in this layer as an array
    l = np.array(df_z[['x', 'y', 'z']].values, dtype='float64')
    n = len(df_z)
    

    # Process all points in the current layer
    initial_point = l[0]  # First point of the layer
    ltp = [pp, initial_point]  # Start the layer path
    

    print(f"Layer points:\n{l}")

    while n > 0:
        # Calculate distances from the current tool position to all remaining points
        distances = np.linalg.norm(l - pp)
        
        print(f"Distances: {distances}")
        
        # Find the closest point
        closest_idx = np.argmin(distances)
        closest_point = l[closest_idx]

        # Add the closest point to the tool path
        ltp.append(closest_point)
        pp = closest_point

        # Remove the processed point
        l = np.delete(l, closest_idx, axis=0)
        n -= 1

    # Close the path for the layer (go back to the initial point of this layer)
    ltp.append(initial_point)
    pp = initial_point  # Update the tool position to the initial point

    # Append the current layer's tool path to the full tool path
    ll.extend(ltp)

    # Remove processed points from df
    df = df[df['z'] < s - tolerance]

    if len(df) > 0:
        # Find the next layer (next highest z)
        next_layer_z = df['z'].max()
        df_next_layer = df[(df['z'] >= next_layer_z - tolerance) & (df['z'] <= next_layer_z + tolerance)]

        # Apply ConvexHull to the next layer
        points_next_layer = df_next_layer[['x', 'y']].values
        hull_next_layer = ConvexHull(points_next_layer)
        df_next_layer = df_next_layer.iloc[hull_next_layer.vertices]

        # Extract coordinates of the next layer's points
        next_layer_coords = df_next_layer[['x', 'y', 'z']].values

        # Calculate distances from the current tool position (pp) to all points in the next layer
        distances_to_next_layer = np.linalg.norm(next_layer_coords - pp, axis=1)

        # Find the closest point in the next layer
        closest_idx_next_layer = np.argmin(distances_to_next_layer)
        closest_point_next_layer = next_layer_coords[closest_idx_next_layer]

    ll.insert(0, closest_point_next_layer)


# Save tool path directly
df_ltp = pd.DataFrame(ll, columns=['x', 'y', 'z'])
df_ltp.to_csv('toolpath.csv', index=False)

# Convert to G-code and write to a text file
df_gchord = pd.DataFrame({
    'G': 'G01',
    'x': 'X' + df_ltp['x'].astype(str),
    'y': 'Y' + df_ltp['y'].astype(str),
    'z': 'Z' + df_ltp['z'].astype(str),
    'EOB': ';'
})
df_gchord.to_csv('Gchord.txt', index=False, sep=" ", header=False)



Layer points:
[[ 5. -4. 17.]
 [ 7. -0. 17.]
 [ 5.  4. 17.]
 [ 4.  5. 17.]
 [ 2.  6. 17.]
 [-2.  6. 17.]
 [-4.  5. 17.]
 [-5.  4. 17.]
 [-7. -0. 17.]
 [-5. -4. 17.]
 [-4. -5. 17.]
 [-2. -6. 17.]
 [ 2. -6. 17.]
 [ 4. -5. 17.]]
Distances: 1433.259222890263
Distances: 34.058772731852805
Distances: 35.38361202590826
Distances: 32.802438933713454
Distances: 32.28002478313795
Distances: 30.95157508108432
Distances: 27.820855486487112
Distances: 25.768197453450252
Distances: 24.20743687382041
Distances: 19.44222209522358
Distances: 12.24744871391589
Distances: 10.295630140987
Distances: 7.280109889280518
Distances: 2.23606797749979
Layer points:
[[  9.  -5.  16.]
 [ 11.  -2.  16.]
 [ 11.   2.  16.]
 [  9.   5.  16.]
 [  5.   9.  16.]
 [  2.  11.  16.]
 [ -2.  11.  16.]
 [ -5.   9.  16.]
 [ -9.   5.  16.]
 [-11.   2.  16.]
 [-11.  -2.  16.]
 [ -9.  -5.  16.]
 [ -5.  -9.  16.]
 [ -2. -11.  16.]
 [  2. -11.  16.]
 [  5.  -9.  16.]]
Distances: 50.19960159204453
Distances: 59.531504264548865
Distan

In [15]:
#No rounding

import pyvista as pv
import csv
import pandas as pd
import numpy as np
from scipy.spatial import ConvexHull

# Load mesh and prepare CSV file of points
mesh = pv.read('freeform_test_7.stl')
points = mesh.points
with open('dots.csv', 'w') as f:
    writer = csv.writer(f)
    writer.writerows(points)

df = pd.read_csv('dots.csv').dropna()
df = df.set_axis(['x', 'y', 'z'], axis='columns')

# Remove points close to the bottom plane
mini = df['z'].min()
df = df[df['z'] > mini + 0.1]
df.to_csv('dots.csv')

# Initial variables
ll = []  # Tool path storage
pp = np.array([0, 0, 400])  # Initial tool position

while len(df) > 0:
    # Identify the highest layer (exact Z value)
    s = df['z'].max()
    df_z = df[df['z'] == s]  # Select points exactly at the highest Z-value

    if len(df_z) < 3:  # Checks if the layer has less than 3 points (ConvexHull requires at least 3 points)
        df = df[df['z'] < s]  # Remove points at the current Z-value
        continue

    # If there are enough points, calculate the convex hull
    points_layer = df_z[['x', 'y']].values

    # Fix: Handle cases where ConvexHull might fail due to collinearity or precision
    if len(points_layer) >= 3 and np.linalg.matrix_rank(points_layer - points_layer.mean(axis=0)) > 1:
        try:
            hull = ConvexHull(points_layer)
            df_z = df_z.iloc[hull.vertices]
        except Exception as e:
            print(f"ConvexHull error: {e}. Using all points instead.")
    else:
        print("Layer points are collinear or insufficient. Skipping ConvexHull.")

    # Store all points in this layer as an array
    l = np.array(df_z[['x', 'y', 'z']].values, dtype='float64')
    n = len(df_z)

    # Process all points in the current layer
    initial_point = l[0]  # First point of the layer
    ltp = [pp.tolist(), initial_point.tolist()]  # Start the layer by visiting the initial point
    pp = initial_point  # Update the tool position to the initial point

    while n > 0:
        # Compute distances and find the closest point
        distances = np.linalg.norm(l - pp, axis=1)
        closest_idx = np.argmin(distances)
        closest_point = l[closest_idx]

        # Add the closest point to the tool path
        ltp.append(closest_point.tolist())
        pp = closest_point

        # Remove the processed point from the list
        l = np.delete(l, closest_idx, axis=0)
        n -= 1

    # Add the initial point to the end of the tool path for the layer
    ltp.append(initial_point.tolist())
    pp = initial_point  # Update the tool position to the initial point

    # Append the current layer's tool path to the full tool path
    ll.extend(ltp)

    # Remove processed points from df
    df = df[df['z'] < s]

# Save the toolpath to a CSV
with open('toolpath.csv', 'w') as f:
    writer = csv.writer(f)
    writer.writerows(ll)

# Process for G-code conversion
df_ltp = pd.DataFrame(ll, columns=['x', 'y', 'z'])
df_ltp.to_csv('toolpath.csv', index=False)

# Convert coordinates to G-code format
l = np.array(ll)
lx, ly, lz = l[:, 0].astype(str), l[:, 1].astype(str), l[:, 2].astype(str)
lx = ['X' + x for x in lx]
ly = ['Y' + y for y in ly]
lz = ['Z' + z for z in lz]

# Write G-code commands to CSV
df_gchord = pd.DataFrame({'G': 'G01', 'x': lx, 'y': ly, 'z': lz, 'EOB': ';'})
with open("Gchord.csv", 'w', newline='') as outf:
    writer = csv.writer(outf)
    writer.writerows(df_gchord.values)

df_g = pd.read_csv('Gchord.csv')
df_g.to_csv('Gchord.txt', index=False, sep=" ")


Layer points are collinear or insufficient. Skipping ConvexHull.
Layer points are collinear or insufficient. Skipping ConvexHull.
Layer points are collinear or insufficient. Skipping ConvexHull.
Layer points are collinear or insufficient. Skipping ConvexHull.
Layer points are collinear or insufficient. Skipping ConvexHull.
Layer points are collinear or insufficient. Skipping ConvexHull.
Layer points are collinear or insufficient. Skipping ConvexHull.
Layer points are collinear or insufficient. Skipping ConvexHull.
Layer points are collinear or insufficient. Skipping ConvexHull.
Layer points are collinear or insufficient. Skipping ConvexHull.
Layer points are collinear or insufficient. Skipping ConvexHull.
Layer points are collinear or insufficient. Skipping ConvexHull.
Layer points are collinear or insufficient. Skipping ConvexHull.
Layer points are collinear or insufficient. Skipping ConvexHull.
Layer points are collinear or insufficient. Skipping ConvexHull.
Layer points are collinea

In [ ]:
#No Tolerance

import pyvista as pv
import csv
import pandas as pd
import numpy as np
from scipy.spatial import ConvexHull

# Load mesh and prepare CSV file of points
mesh = pv.read('freeform_test_7.stl')
points = mesh.points
with open('dots.csv', 'w') as f:
    writer = csv.writer(f)
    writer.writerows(points)

df = pd.read_csv('dots.csv').dropna()
df = df.set_axis(['x', 'y', 'z'], axis='columns')

# Remove points close to the bottom plane
mini = df['z'].min()
df = df[df['z'] > mini + 0.1]
df.to_csv('dots.csv')

# Initial variables
ll = []  # Tool path storage
pp = [0, 0, 400]  # Initial tool position

# Round values for layer selection
df['x'] = df['x'].round(2)
df['y'] = df['y'].round(2)

while len(df) > 0:
    # Identify the highest layer
    s = df['z'].max()
    df_z = df[df['z'] == s]  # Select points exactly at the highest Z value

    if len(df_z) < 3:  # Checks if the layer has less than 3 points
        df = df[df['z'] < s]
        continue

    # Get unique 2D points for the convex hull
    points_layer = np.unique(df_z[['x', 'y']].values, axis=0)

    if len(points_layer) < 3:  # Skip if fewer than 3 unique points
        df = df[df['z'] < s]
        continue

    # Try to calculate the convex hull
    try:
        hull = ConvexHull(points_layer)
        df_z = df_z.iloc[hull.vertices]
    except:
        df = df[df['z'] < s]  # Skip this layer if ConvexHull fails
        continue


    # Store all points in this layer as an array
    l = np.array(df_z[['x', 'y', 'z']].values, dtype='float64')
    n = len(df_z)

    # Process all points in the current layer
    initial_point = l[0]  # First point of the layer
    ltp = [pp, initial_point.tolist()]  # Start the layer by visiting the initial point
    pp = initial_point  # Update the tool position to the initial point

    while n > 0:
        distances = np.linalg.norm(l - pp, axis=1)
        closest_idx = np.argmin(distances)
        closest_point = l[closest_idx]

        # Add the closest point to the tool path
        ltp.append(closest_point.tolist())
        pp = closest_point

        # Remove the processed point from the list
        l = np.delete(l, closest_idx, axis=0)
        n -= 1

    ltp.append(initial_point.tolist())
    pp = initial_point  # Update the tool position to the initial point

    # Append the current layer's tool path to the full tool path
    ll.extend(ltp)

    # Remove processed points from df
    df = df[df['z'] < s]

# Save the toolpath to a CSV
with open('toolpath.csv', 'w') as f:
    writer = csv.writer(f)
    writer.writerows(ll)

# Process for G-code conversion
df_ltp = pd.DataFrame(ll, columns=['x', 'y', 'z'])
df_ltp.to_csv('toolpath.csv', index=False)

# Convert coordinates to G-code format
l = np.array(ll)
lx, ly, lz = l[:, 0].astype(str), l[:, 1].astype(str), l[:, 2].astype(str)
lx = ['X' + x for x in lx]
ly = ['Y' + y for y in ly]
lz = ['Z' + z for z in lz]

# Write G-code commands to CSV
df_gchord = pd.DataFrame({'G': 'G01', 'x': lx, 'y': ly, 'z': lz, 'EOB': ';'})
with open("Gchord.csv", 'w', newline='') as outf:
    writer = csv.writer(outf)
    writer.writerows(df_gchord.values)

df_g = pd.read_csv('Gchord.csv')
df_g.to_csv('Gchord.txt', index=False, sep=" ")


In [69]:
#11/25 Code by the end of the day, No "jumping" however when the tool path moves to the next layer it goes straight down then process the layer
# Need to make sure that when the tool path moves from layer to layer it is smooth and not straight down

import pyvista as pv
import csv
import pandas as pd
import numpy as np
from scipy.spatial import ConvexHull
pd.options.mode.copy_on_write = True


# Load mesh and prepare CSV file of points
mesh = pv.read('freeform_test_7.stl')
points = mesh.points
with open('dots.csv', 'w') as f:
    writer = csv.writer(f)
    writer.writerows(points)

df = pd.read_csv('dots.csv').dropna()
df = df.set_axis(['x', 'y', 'z'], axis='columns')

# Remove points close to the bottom plane
df = df[df['z'] > df['z'].min() + 0.1]

# Initial variables
ll = []  # Tool path storage
pp = np.array([0, 0, 400])  # Initial tool position
tolerance = 0.1  # Z tolerance for grouping points into layers

# Round values for layer selection
df['z'] = df['z'].round(0)
df['x'] = df['x'].round(0)
df['y'] = df['y'].round(0)

while len(df) > 0:
    # Identify the highest layer
    current_layer_z = df['z'].max()
    current_layer = df[(df['z'] >= current_layer_z - tolerance) & (df['z'] <= current_layer_z + tolerance)]


    # Skip layers with fewer than 3 points
    if len(current_layer) < 3:
        df = df[df['z'] < current_layer_z - tolerance]
        continue

    # Filter points to retain only boundary points using ConvexHull
    layer_points = current_layer[['x', 'y']].values
    hull = ConvexHull(layer_points)
    current_layer = current_layer.iloc[hull.vertices]


    # Convert to NumPy array for calculations
    current_coords = np.array(current_layer[['x', 'y', 'z']].values)
    
    # Process points in the current layer
    layer_path = []
    while len(current_coords) > 0:
        # Calculate distances from current position to all points in the layer
        distances = np.linalg.norm(current_coords - pp, axis=1)
        closest_idx = np.argmin(distances)
        closest_point = current_coords[closest_idx]

        # Append the closest point and update current position
        layer_path.append(closest_point)
        pp = closest_point

        # Remove the processed point
        current_coords = np.delete(current_coords, closest_idx, axis=0)

    # Close the loop for the current layer
    layer_path.append(layer_path[0])  # Return to the first point
    pp = layer_path[0]  # Update tool position

    # Append the layer path to the overall tool path
    ll.extend(layer_path)

    # Remove processed points from the main DataFrame
    df = df[df['z'] < current_layer_z - tolerance]

    # Find the closest point in the next layer if it exists
    if len(df) > 0:
        next_layer_z = df['z'].max()
        next_layer = df[(df['z'] >= next_layer_z - tolerance) & (df['z'] <= next_layer_z + tolerance)]
        
        # Update the tool position to the closest point on the next layer
        next_layer_coords = np.array(next_layer[['x', 'y', 'z']].values)
        distances_to_next_layer = np.linalg.norm(next_layer_coords - pp, axis=1)
        closest_next_idx = np.argmin(distances_to_next_layer)
        closest_point_next_layer = next_layer_coords[closest_next_idx]
        pp = closest_point_next_layer  # Update the tool position
        ll.append(closest_point_next_layer)


# Save tool path
df_ltp = pd.DataFrame(ll, columns=['x', 'y', 'z'])
df_ltp.to_csv('toolpath.csv', index=False)

# Convert to G-code and save
df_gchord = pd.DataFrame({
    'G': 'G01',
    'x': 'X' + df_ltp['x'].astype(str),
    'y': 'Y' + df_ltp['y'].astype(str),
    'z': 'Z' + df_ltp['z'].astype(str),
    'EOB': ';'
})
df_gchord.to_csv('Gchord.txt', index=False, sep=" ", header=False)




In [68]:
#shortening segments of the convexhull

import pyvista as pv
import csv
import pandas as pd
import numpy as np
from scipy.spatial import ConvexHull
pd.options.mode.copy_on_write = True


# Load mesh and prepare CSV file of points
mesh = pv.read('freeform_test_7.stl')
points = mesh.points
with open('dots.csv', 'w') as f:
    writer = csv.writer(f)
    writer.writerows(points)

df = pd.read_csv('dots.csv').dropna()
df = df.set_axis(['x', 'y', 'z'], axis='columns')

# Remove points close to the bottom plane
df = df[df['z'] > df['z'].min() + 0.1]

# Initial variables
ll = []  # Tool path storage
pp = np.array([0, 0, 400])  # Initial tool position
tolerance = 0.1  # Z tolerance for grouping points into layers

# Round values for layer selection
df['z'] = df['z'].round(0)
df['x'] = df['x'].round(0)
df['y'] = df['y'].round(0)

# Function to subdivide edges into smaller segments
def subdivide_edges(boundary_points, subdivisions=5):
    subdivided_points = []
    for i in range(len(boundary_points)):
        point_a = boundary_points[i]
        point_b = boundary_points[(i + 1) % len(boundary_points)]  # Loop back to the first point
        
        for j in range(subdivisions):
            # Linear interpolation between point_a and point_b
            new_point = point_a + (j / subdivisions) * (point_b - point_a)
            subdivided_points.append(new_point)

    return np.array(subdivided_points)

while len(df) > 0:
    # Identify the highest layer
    current_layer_z = df['z'].max()
    current_layer = df[(df['z'] >= current_layer_z - tolerance) & (df['z'] <= current_layer_z + tolerance)]

    # Skip layers with fewer than 3 points
    if len(current_layer) < 3:
        df = df[df['z'] < current_layer_z - tolerance]
        continue

    # Filter points to retain only boundary points using ConvexHull
    layer_points = current_layer[['x', 'y']].values
    hull = ConvexHull(layer_points)
    boundary_points = layer_points[hull.vertices]

    # Subdivide edges of the convex hull to get more boundary points
    subdivided_boundary_points = subdivide_edges(boundary_points, subdivisions=5)

    # Convert to NumPy array for calculations
    current_layer_coords = np.array(current_layer[['x', 'y', 'z']].values)
    
    # Process points in the current layer
    layer_path = []
    while len(current_layer_coords) > 0:
        # Calculate distances from current position to all points in the layer
        distances = np.linalg.norm(current_layer_coords - pp, axis=1)
        closest_idx = np.argmin(distances)
        closest_point = current_layer_coords[closest_idx]

        # Append the closest point and update current position
        layer_path.append(closest_point)
        pp = closest_point

        # Remove the processed point
        current_layer_coords = np.delete(current_layer_coords, closest_idx, axis=0)

    # Close the loop for the current layer
    layer_path.append(layer_path[0])  # Return to the first point
    pp = layer_path[0]  # Update tool position

    # Append the layer path to the overall tool path
    ll.extend(layer_path)

    # Remove processed points from the main DataFrame
    df = df[df['z'] < current_layer_z - tolerance]

    if len(df) > 0:
        next_layer_z = df['z'].max()
        next_layer = df[(df['z'] >= next_layer_z - tolerance) & (df['z'] <= next_layer_z + tolerance)]
        
        # Extract points for the next layer
        next_layer_coords = np.array(next_layer[['x', 'y', 'z']].values)
        
        # Calculate the Convex Hull for the next layer
        if len(next_layer_coords) >= 3:  # Convex Hull requires at least 3 points
            hull = ConvexHull(next_layer_coords[:, :2])  # Use x, y coordinates only
            convex_hull_points = next_layer_coords[hull.vertices]  # Select points on the hull
            
            # Subdivide the edges for the next layer's convex hull
            subdivided_next_layer_points = subdivide_edges(convex_hull_points, subdivisions=5)
        else:
            subdivided_next_layer_points = next_layer_coords  # If fewer than 3 points, use all points

        # Find the closest point on the Convex Hull to the current position
        distances_to_hull = np.linalg.norm(subdivided_next_layer_points - pp, axis=1)
        closest_hull_idx = np.argmin(distances_to_hull)
        closest_point_hull = subdivided_next_layer_points[closest_hull_idx]

        # Update the tool position and append the point to the toolpath
        pp = closest_point_hull
        ll.append(closest_point_hull)

# Save tool path
df_ltp = pd.DataFrame(ll, columns=['x', 'y', 'z'])
df_ltp.to_csv('toolpath.csv', index=False)

# Convert to G-code and save
df_gchord = pd.DataFrame({
    'G': 'G01',
    'x': 'X' + df_ltp['x'].astype(str),
    'y': 'Y' + df_ltp['y'].astype(str),
    'z': 'Z' + df_ltp['z'].astype(str),
    'EOB': ';'
})
df_gchord.to_csv('Gchord.txt', index=False, sep=" ", header=False)


In [ ]:
#attempt on using alpha shape instead of convexhull

import pyvista as pv
import csv
import pandas as pd
import numpy as np
from alphashape import alphashape
from shapely.geometry import Point
pd.options.mode.copy_on_write = True

# Load mesh and prepare CSV file of points
mesh = pv.read('freeform_test_7.stl')
points = mesh.points
with open('dots.csv', 'w') as f:
    writer = csv.writer(f)
    writer.writerows(points)

df = pd.read_csv('dots.csv').dropna()
df = df.set_axis(['x', 'y', 'z'], axis='columns')

# Remove points close to the bottom plane
df = df[df['z'] > df['z'].min() + 0.1]

# Initial variables
ll = []  # Tool path storage
pp = np.array([0, 0, 400])  # Initial tool position
tolerance = 0.1  # Z tolerance for grouping points into layers

# Round values for layer selection
df['z'] = df['z'].round(1)
df['x'] = df['x'].round(0)
df['y'] = df['y'].round(0)

while len(df) > 0:
    # Identify the highest layer
    current_layer_z = df['z'].max()
    current_layer = df[(df['z'] >= current_layer_z - tolerance) & (df['z'] <= current_layer_z + tolerance)]

    print(f"Processing layer at z = {current_layer_z}")
    if len(current_layer) < 3:
        print(f"Skipping layer at z = {current_layer_z} due to insufficient points.")
        df = df[df['z'] < current_layer_z - tolerance]
        continue

    # Alpha shape boundary detection
    layer_points = current_layer[['x', 'y']].values
    alpha_shape = alphashape(layer_points, alpha=5.0)

    if alpha_shape is None or alpha_shape.is_empty:
        print(f"Alpha shape is invalid for layer at z = {current_layer_z}")
        boundary_points = layer_points
    else:
        boundary_points = np.array([
            point for point in layer_points
            if alpha_shape.contains(Point(point))
        ])

    if len(boundary_points) == 0:
        print(f"No boundary points found for layer at z = {current_layer_z}")
        df = df[df['z'] < current_layer_z - tolerance]
        continue

    # Update current_layer with boundary points
    current_layer = current_layer[
        current_layer[['x', 'y']].apply(tuple, axis=1).isin(map(tuple, boundary_points))
    ]

    current_coords = np.array(current_layer[['x', 'y', 'z']].values)
    if len(current_coords) == 0:
        print(f"No valid points in layer at z = {current_layer_z}")
        df = df[df['z'] < current_layer_z - tolerance]
        continue

    # Process points in the current layer
    layer_path = []
    while len(current_coords) > 0:
        distances = np.linalg.norm(current_coords - pp, axis=1)
        closest_idx = np.argmin(distances)
        closest_point = current_coords[closest_idx]

        layer_path.append(closest_point)
        pp = closest_point
        current_coords = np.delete(current_coords, closest_idx, axis=0)

    # Close the layer loop
    if len(layer_path) > 0:
        layer_path.append(layer_path[0])
        pp = layer_path[0]
    ll.extend(layer_path)

    # Remove processed points
    processed_z_values = current_layer['z'].unique()
    df = df[~df['z'].isin(processed_z_values)]

    print(f"Remaining points in df: {len(df)}")

    # Smooth transition to the next layer
    if len(df) > 0:
        next_layer_z = df['z'].max()
        next_layer = df[(df['z'] >= next_layer_z - tolerance) & (df['z'] <= next_layer_z + tolerance)]

        # Extract points for the next layer
        next_layer_coords = np.array(next_layer[['x', 'y', 'z']].values)

        # Skip if no valid points in the next layer
        if len(next_layer_coords) == 0:
            continue

        # Find the closest point in the next layer to the current position
        distances_to_next_layer = np.linalg.norm(next_layer_coords - pp, axis=1)
        closest_next_idx = np.argmin(distances_to_next_layer)
        closest_point_next_layer = next_layer_coords[closest_next_idx]

        # Smoothly transition to the next layer by directly moving to the closest point
        pp = closest_point_next_layer
        ll.append(closest_point_next_layer)

# Save tool path
df_ltp = pd.DataFrame(ll, columns=['x', 'y', 'z'])
df_ltp.to_csv('toolpath.csv', index=False)

# Convert to G-code and save
df_gchord = pd.DataFrame({
    'G': 'G01',
    'x': 'X' + df_ltp['x'].astype(str),
    'y': 'Y' + df_ltp['y'].astype(str),
    'z': 'Z' + df_ltp['z'].astype(str),
    'EOB': ';'
})
df_gchord.to_csv('Gchord.txt', index=False, sep=" ", header=False)


Processing layer at z = 17.2
Alpha shape is invalid for layer at z = 17.2
Remaining points in df: 2751
Processing layer at z = 17.0
Alpha shape is invalid for layer at z = 17.0
Remaining points in df: 2703
Processing layer at z = 16.8
Alpha shape is invalid for layer at z = 16.8
Remaining points in df: 2661
Processing layer at z = 16.6
Alpha shape is invalid for layer at z = 16.6
Remaining points in df: 2609
Processing layer at z = 16.4
Alpha shape is invalid for layer at z = 16.4
Remaining points in df: 2567
Processing layer at z = 16.2
Alpha shape is invalid for layer at z = 16.2
Remaining points in df: 2521
Processing layer at z = 16.0
Alpha shape is invalid for layer at z = 16.0
Remaining points in df: 2471
Processing layer at z = 15.8
Alpha shape is invalid for layer at z = 15.8
Remaining points in df: 2447
Processing layer at z = 15.7
Alpha shape is invalid for layer at z = 15.7
Remaining points in df: 2405
Processing layer at z = 15.5
Alpha shape is invalid for layer at z = 15.5

In [ ]:
#11/27 Solved all the problems that I had before, the tool path travels smoothly to the next layer
#next problem, for data is rounding NOT to a whole number, problems arive (need to find a way to add more detail while ensuring that the tool path still remains clean like the whole number tool path)

import pyvista as pv
import csv
import pandas as pd
import numpy as np
from scipy.spatial import ConvexHull
pd.options.mode.copy_on_write = True


# Load mesh and prepare CSV file of points
mesh = pv.read('freeform_test_7_pt1.stl')
points = mesh.points
with open('dots.csv', 'w') as f:
    writer = csv.writer(f)
    writer.writerows(points)

df = pd.read_csv('dots.csv').dropna()
df = df.set_axis(['x', 'y', 'z'], axis='columns')

# Remove points close to the bottom plane
df = df[df['z'] > df['z'].min() + 0.1]

# Initial variables
ll = []  # Tool path storage
pp = np.array([0, 0, 400])  # Initial tool position
tolerance = 0.33  # Z tolerance for grouping points into layers  

# # Round values for layer selection
# # Round z to the nearest half or whole number
# df['z'] = (df['z'].round() * 3) / 3
# # df['z'] = df['z'].round()
# df['x'] = df['x'].round()
# df['y'] = df['y'].round()

while len(df) > 0:
    # Identify the highest layer
    current_layer_z = df['z'].max()
    current_layer = df[(df['z'] >= current_layer_z - tolerance) & (df['z'] <= current_layer_z + tolerance)]


    # Skip layers with fewer than 3 points
    if len(current_layer) < 3:
        df = df[df['z'] < current_layer_z - tolerance]
        continue

    # Filter points to retain only boundary points using ConvexHull
    layer_points = current_layer[['x', 'y']].values
    hull = ConvexHull(layer_points)
    current_layer = current_layer.iloc[hull.vertices]

    # Convert to NumPy array for calculations
    current_coords = np.array(current_layer[['x', 'y', 'z']].values)

    # Process points in the current layer
    layer_path = []
    while len(current_coords) > 0:
        # Calculate distances from current position to all points in the layer
        distances = np.linalg.norm(current_coords - pp, axis=1)
        closest_idx = np.argmin(distances)
        closest_point = current_coords[closest_idx]

        # Append the closest point and update current position
        layer_path.append(closest_point)
        pp = closest_point

        # Remove the processed point
        current_coords = np.delete(current_coords, closest_idx, axis=0)

    # Close the loop for the current layer
    layer_path.append(layer_path[0])  # Return to the first point
    pp = layer_path[0]  # Update tool position

    # Append the layer path to the overall tool path
    ll.extend(layer_path)

    # Remove processed points from the main DataFrame
    df = df[df['z'] < current_layer_z - tolerance]

    # Find the closest point in the next layer if it exists
    if len(df) > 0:
        next_layer_z = df['z'].max()
        next_layer = df[(df['z'] >= next_layer_z - tolerance) & (df['z'] <= next_layer_z + tolerance)]
        
        # Extract points for the next layer
        next_layer_coords = np.array(next_layer[['x', 'y', 'z']].values)
        
        # Calculate the Convex Hull for the next layer
        if len(next_layer_coords) >= 3:  # Convex Hull requires at least 3 points
            hull = ConvexHull(next_layer_coords[:, :2])  # Use x, y coordinates only
            convex_hull_points = next_layer_coords[hull.vertices]  # Select points on the hull
        else:
            convex_hull_points = next_layer_coords  # If fewer than 3 points, use all points

        # Find the closest point on the Convex Hull to the current position
        distances_to_hull = np.linalg.norm(convex_hull_points - pp, axis=1)
        closest_hull_idx = np.argmin(distances_to_hull)
        closest_point_hull = convex_hull_points[closest_hull_idx]

        # Update the tool position and append the point to the toolpath
        pp = closest_point_hull
        ll.append(closest_point_hull)


# Save tool path
df_ltp = pd.DataFrame(ll, columns=['x', 'y', 'z'])

# # Flip the tool path by reversing the order of the points
# df_ltp = df_ltp.iloc[::-1].reset_index(drop=True)

# Save the flipped tool path
df_ltp.to_csv('toolpath.csv', index=False)

# Convert to G-code and save
df_gchord = pd.DataFrame({
    'G': 'G01',
    'x': 'X' + df_ltp['x'].astype(str),
    'y': 'Y' + df_ltp['y'].astype(str),
    'z': 'Z' + df_ltp['z'].astype(str),
    'EOB': ';'
})
df_gchord.to_csv('Gchord.txt', index=False, sep=" ", header=False)





In [120]:
#Edge subdivision
import pyvista as pv
import csv
import pandas as pd
import numpy as np
from scipy.spatial import ConvexHull
pd.options.mode.copy_on_write = True


# Load mesh and prepare CSV file of points
mesh = pv.read('freeform_test_7.stl')
points = mesh.points
with open('dots.csv', 'w') as f:
    writer = csv.writer(f)
    writer.writerows(points)

df = pd.read_csv('dots.csv').dropna()
df = df.set_axis(['x', 'y', 'z'], axis='columns')

# Remove points close to the bottom plane
df = df[df['z'] > df['z'].min() + 0.1]

# Initial variables
ll = []  # Tool path storage
pp = np.array([0, 0, 400])  # Initial tool position
tolerance = 0.1  # Z tolerance for grouping points into layers

# Round values for layer selection
df['z'] = df['z'].round(0)
df['x'] = df['x'].round(0)
df['y'] = df['y'].round(0)

subdivisions=1  #This is for the subdivisions along the convexhull, change this for smoothness/more detail for the shape

# Function to subdivide edges into smaller segments
def subdivide_edges(boundary_points, subdivisions):
    subdivided_points = []
    for i in range(len(boundary_points)):
        point_a = boundary_points[i]
        point_b = boundary_points[(i + 1) % len(boundary_points)]  # Loop back to the first point
        
        for j in range(subdivisions):
            # Linear interpolation between point_a and point_b
            new_point = point_a + (j / subdivisions) * (point_b - point_a)
            subdivided_points.append(new_point)

    return np.array(subdivided_points)

while len(df) > 0:
    # Identify the highest layer
    current_layer_z = df['z'].max()
    current_layer = df[(df['z'] >= current_layer_z - tolerance) & (df['z'] <= current_layer_z + tolerance)]

    # Skip layers with fewer than 3 points
    if len(current_layer) < 3:
        df = df[df['z'] < current_layer_z - tolerance]
        continue

    # Filter points to retain only boundary points using ConvexHull
    layer_points = current_layer[['x', 'y']].values
    hull = ConvexHull(layer_points)
    boundary_points = layer_points[hull.vertices]

    # Subdivide edges of the convex hull to get more boundary points
    subdivided_boundary_points = subdivide_edges(boundary_points, subdivisions)

    # Convert to NumPy array for calculations
    current_layer_coords = np.array(current_layer[['x', 'y', 'z']].values)
    
    # Process points in the current layer
    layer_path = []
    while len(current_layer_coords) > 0:
        # Calculate distances from current position to all points in the layer
        distances = np.linalg.norm(current_layer_coords - pp, axis=1)
        closest_idx = np.argmin(distances)
        closest_point = current_layer_coords[closest_idx]

        # Append the closest point and update current position
        layer_path.append(closest_point)
        pp = closest_point

        # Remove the processed point
        current_layer_coords = np.delete(current_layer_coords, closest_idx, axis=0)

    # Close the loop for the current layer
    layer_path.append(layer_path[0])  # Return to the first point
    pp = layer_path[0]  # Update tool position

    # Append the layer path to the overall tool path
    ll.extend(layer_path)

    # Remove processed points from the main DataFrame
    df = df[df['z'] < current_layer_z - tolerance]

    if len(df) > 0:
        next_layer_z = df['z'].max()
        next_layer = df[(df['z'] >= next_layer_z - tolerance) & (df['z'] <= next_layer_z + tolerance)]
        
        # Extract points for the next layer
        next_layer_coords = np.array(next_layer[['x', 'y', 'z']].values)
        
        # Calculate the Convex Hull for the next layer
        if len(next_layer_coords) >= 3:  # Convex Hull requires at least 3 points
            hull = ConvexHull(next_layer_coords[:, :2])  # Use x, y coordinates only
            convex_hull_points = next_layer_coords[hull.vertices]  # Select points on the hull
            
            # Subdivide the edges for the next layer's convex hull
            subdivided_next_layer_points = subdivide_edges(convex_hull_points, subdivisions)
        else:
            subdivided_next_layer_points = next_layer_coords  # If fewer than 3 points, use all points

        # Find the closest point on the Convex Hull to the current position
        distances_to_hull = np.linalg.norm(subdivided_next_layer_points - pp, axis=1)
        closest_hull_idx = np.argmin(distances_to_hull)
        closest_point_hull = subdivided_next_layer_points[closest_hull_idx]

        # Update the tool position and append the point to the toolpath
        pp = closest_point_hull
        ll.append(closest_point_hull)

# Save tool path
df_ltp = pd.DataFrame(ll, columns=['x', 'y', 'z'])
df_ltp.to_csv('toolpath.csv', index=False)

# Convert to G-code and save
df_gchord = pd.DataFrame({
    'G': 'G01',
    'x': 'X' + df_ltp['x'].astype(str),
    'y': 'Y' + df_ltp['y'].astype(str),
    'z': 'Z' + df_ltp['z'].astype(str),
    'EOB': ';'
})
df_gchord.to_csv('Gchord.txt', index=False, sep=" ", header=False)


In [ ]:
#Edge subdivision and inflated margins
import pyvista as pv
import csv
import pandas as pd
import numpy as np
from scipy.spatial import ConvexHull
pd.options.mode.copy_on_write = True


# Load mesh and prepare CSV file of points
mesh = pv.read('freeform_test_7.stl')
points = mesh.points
with open('dots.csv', 'w') as f:
    writer = csv.writer(f)
    writer.writerows(points)

df = pd.read_csv('dots.csv').dropna()
df = df.set_axis(['x', 'y', 'z'], axis='columns')

# Remove points close to the bottom plane
df = df[df['z'] > df['z'].min() + 0.1]

# Initial variables
ll = []  # Tool path storage
pp = np.array([0, 0, 400])  # Initial tool position
tolerance = 0.1  # Z tolerance for grouping points into layers
margin = 0.1  # Inflation margin (distance to offset points outward)

subdivisions=10  # This is for the subdivisions along the convex hull, change this for smoothness/more detail for the shape

# Round values for layer selection
df['z'] = df['z'].round(0)
df['x'] = df['x'].round(0)
df['y'] = df['y'].round(0)


# Function to subdivide edges into smaller segments
def subdivide_edges(boundary_points, subdivisions):
    subdivided_points = []
    for i in range(len(boundary_points)):
        point_a = boundary_points[i]
        point_b = boundary_points[(i + 1) % len(boundary_points)]  # Loop back to the first point
        
        for j in range(subdivisions):
            # Linear interpolation between point_a and point_b
            new_point = point_a + (j / subdivisions) * (point_b - point_a)
            subdivided_points.append(new_point)

    return np.array(subdivided_points)

# Function to inflate boundary points by the margin factor
def inflate_boundary_points(boundary_points, margin):
    # Calculate the center of the convex hull
    center = np.mean(boundary_points, axis=0)
    
    # Inflate each boundary point outward from the center
    inflated_points = []
    for point in boundary_points:
        direction = point - center  # Vector from center to the point
        direction_normalized = direction / np.linalg.norm(direction)  # Normalize the direction
        inflated_point = point + direction_normalized * margin  # Offset the point by the margin
        inflated_points.append(inflated_point)
    
    return np.array(inflated_points)

while len(df) > 0:
    # Identify the highest layer
    current_layer_z = df['z'].max()
    current_layer = df[(df['z'] >= current_layer_z - tolerance) & (df['z'] <= current_layer_z + tolerance)]

    # Skip layers with fewer than 3 points
    if len(current_layer) < 3:
        df = df[df['z'] < current_layer_z - tolerance]
        continue

    # Filter points to retain only boundary points using ConvexHull
    layer_points = current_layer[['x', 'y']].values
    hull = ConvexHull(layer_points)
    boundary_points = layer_points[hull.vertices]

    # Inflate boundary points by the margin
    inflated_boundary_points = inflate_boundary_points(boundary_points, margin)

    # Subdivide edges of the convex hull to get more boundary points
    subdivided_boundary_points = subdivide_edges(inflated_boundary_points, subdivisions)

    # Convert to NumPy array for calculations
    current_layer_coords = np.array(current_layer[['x', 'y', 'z']].values)
    
    # Process points in the current layer
    layer_path = []
    while len(current_layer_coords) > 0:
        # Calculate distances from current position to all points in the layer
        distances = np.linalg.norm(current_layer_coords - pp, axis=1)
        closest_idx = np.argmin(distances)
        closest_point = current_layer_coords[closest_idx]

        # Append the closest point and update current position
        layer_path.append(closest_point)
        pp = closest_point

        # Remove the processed point
        current_layer_coords = np.delete(current_layer_coords, closest_idx, axis=0)

    # Close the loop for the current layer
    layer_path.append(layer_path[0])  # Return to the first point
    pp = layer_path[0]  # Update tool position

    # Append the layer path to the overall tool path
    ll.extend(layer_path)

    # Remove processed points from the main DataFrame
    df = df[df['z'] < current_layer_z - tolerance]

    if len(df) > 0:
        next_layer_z = df['z'].max()
        next_layer = df[(df['z'] >= next_layer_z - tolerance) & (df['z'] <= next_layer_z + tolerance)]
        
        # Extract points for the next layer
        next_layer_coords = np.array(next_layer[['x', 'y', 'z']].values)
        
        # Calculate the Convex Hull for the next layer
        if len(next_layer_coords) >= 3:  # Convex Hull requires at least 3 points
            hull = ConvexHull(next_layer_coords[:, :2])  # Use x, y coordinates only
            convex_hull_points = next_layer_coords[hull.vertices]  # Select points on the hull
            
            # Inflate the convex hull points by the margin
            inflated_next_layer_points = inflate_boundary_points(convex_hull_points, margin)
            
            # Subdivide the edges for the next layer's inflated convex hull
            subdivided_next_layer_points = subdivide_edges(inflated_next_layer_points, subdivisions)
        else:
            subdivided_next_layer_points = next_layer_coords  # If fewer than 3 points, use all points

        # Find the closest point on the Convex Hull to the current position
        distances_to_hull = np.linalg.norm(subdivided_next_layer_points - pp, axis=1)
        closest_hull_idx = np.argmin(distances_to_hull)
        closest_point_hull = subdivided_next_layer_points[closest_hull_idx]

        # Update the tool position and append the point to the toolpath
        pp = closest_point_hull
        ll.append(closest_point_hull)

# Save tool path
df_ltp = pd.DataFrame(ll, columns=['x', 'y', 'z'])
df_ltp.to_csv('toolpath.csv', index=False)

# Convert to G-code and save
df_gchord = pd.DataFrame({
    'G': 'G01',
    'x': 'X' + df_ltp['x'].astype(str),
    'y': 'Y' + df_ltp['y'].astype(str),
    'z': 'Z' + df_ltp['z'].astype(str),
    'EOB': ';'
})
df_gchord.to_csv('Gchord.txt', index=False, sep=" ", header=False)


In [13]:
import pyvista as pv
import csv
import pandas as pd
import numpy as np
import alphashape
from shapely.geometry import Polygon, MultiPolygon, MultiPoint

pd.options.mode.copy_on_write = True

# Load mesh and prepare CSV file of points
mesh = pv.read('freeform_test_7_pt1.stl')
points = mesh.points
with open('dots.csv', 'w') as f:
    writer = csv.writer(f)
    writer.writerows(points)

df = pd.read_csv('dots.csv').dropna()
df = df.set_axis(['x', 'y', 'z'], axis='columns')

# Remove points close to the bottom plane
df = df[df['z'] > df['z'].min() + 0.1]

# Initial variables
ll = []  # Tool path storage

tolerance = 0.1  # Z tolerance for grouping points into layers
alpha_value = 0.5  # Alpha shape flexibility
min_layer_distance = 0.5  # Minimum distance between layers to prevent jumps
max_xy_distance = 0.5  # Maximum allowable distance between points in x and y coordinates

# Function to detect degenerate (collinear or coplanar) points
def is_degenerate(points, tolerance=1e-4):
    """
    Check if the points are degenerate (e.g., collinear or coplanar).
    :param points: Array of points with shape (n, 3).
    :param tolerance: Tolerance for determining degeneracy.
    :return: True if points are degenerate, False otherwise.
    """
    if len(points) < 3:
        return True  # Not enough points to form a valid shape

    # Check if the points are collinear or coplanar by calculating the rank of the matrix
    if np.linalg.matrix_rank(points) < 3:
        return True  # Points are degenerate (e.g., collinear or coplanar)
    
    return False

# Function to check if the points form a valid 2D shape (using shapely)
def valid_polygon_2d(points):
    """
    Check if the 2D points form a valid polygon using shapely.
    :param points: Array of points with shape (n, 2).
    :return: True if the points form a valid polygon, False otherwise.
    """
    try:
        multipoint = MultiPoint(points)
        hull = multipoint.convex_hull  # Get the convex hull of the points
        if isinstance(hull, Polygon) and len(hull.exterior.coords) >= 3:
            return True  # Valid polygon
        else:
            return False  # Invalid polygon (e.g., collinear or degenerate)
    except Exception as e:
        return False  # Error case (invalid input)

# Function to calculate the distance in the x and y plane
def distance_in_xy(p1, p2):
    return np.linalg.norm(p1[:2] - p2[:2])  # Only consider x and y distance

# Check if the distance to the next layer is too large in the x and y plane
def is_jump_in_xy(current_point, next_point, threshold=max_xy_distance):
    return distance_in_xy(current_point, next_point) > threshold

# Identify the highest layer
initial_layer_z = df['z'].max()
initial_layer = df[df['z'] >= initial_layer_z - tolerance]

# Choose the first point in the highest layer as the starting point
initial_point = initial_layer.iloc[0]
pp = initial_point[['x', 'y', 'z']].values

while len(df) > 0:
    # Identify the highest layer
    current_layer_z = df['z'].max()
    current_layer = df[(df['z'] >= current_layer_z - tolerance) & (df['z'] <= current_layer_z + tolerance)]

    # Skip layers with fewer than 3 points
    if len(current_layer) < 3:
        df = df[df['z'] < current_layer_z - tolerance]
        continue

    # Check if the current layer's points are degenerate (collinear or coplanar)
    if is_degenerate(current_layer[['x', 'y', 'z']].values):
        print(f"Degenerate layer detected at z={current_layer_z}, skipping.")
        df = df[df['z'] < current_layer_z - tolerance]
        continue

    # Extract the 2D points (x, y) for Alpha Shape calculation
    layer_points = current_layer[['x', 'y']].values

    # Check if the 2D points form a valid polygon
    if not valid_polygon_2d(layer_points):
        print(f"Invalid 2D points at z={current_layer_z}, skipping Alpha Shape.")
        df = df[df['z'] < current_layer_z - tolerance]
        continue

    # Filter points using Alpha Shape
    boundary = alphashape.alphashape(layer_points, alpha_value)

    boundary_points_2d = []
    if isinstance(boundary, Polygon):
        boundary_points_2d = np.array(boundary.exterior.coords)
    elif isinstance(boundary, MultiPolygon):
        for poly in boundary.geoms:
            if poly.exterior:
                boundary_points_2d.extend(poly.exterior.coords)
        boundary_points_2d = np.array(boundary_points_2d)
    else:
        boundary_points_2d = layer_points

    # Add z-coordinate back and update the DataFrame
    boundary_points = np.column_stack((boundary_points_2d, np.full(len(boundary_points_2d), current_layer_z)))
    current_layer = pd.DataFrame(boundary_points, columns=['x', 'y', 'z'])

    # Convert to NumPy array for calculations
    current_coords = np.array(current_layer[['x', 'y', 'z']].values)

    # Process points in the current layer with optimized nearest-neighbor sorting
    layer_path = []
    while len(current_coords) > 0:
        distances = np.linalg.norm(current_coords - pp, axis=1)
        closest_idx = np.argmin(distances)
        closest_point = current_coords[closest_idx]

        # Append the closest point and update current position
        layer_path.append(closest_point)
        pp = closest_point

        # Remove the processed point
        current_coords = np.delete(current_coords, closest_idx, axis=0)

    # Close the loop for the current layer
    layer_path.append(layer_path[0])  # Return to the first point
    pp = layer_path[0]  # Update tool position

    # Append the layer path to the overall tool path
    ll.extend(layer_path)

    # Remove processed points from the main DataFrame
    df = df[df['z'] < current_layer_z - tolerance]

    # Process the next layer if it exists
    if len(df) > 0:
        next_layer_z = df['z'].max()
        next_layer = df[(df['z'] >= next_layer_z - tolerance) & (df['z'] <= next_layer_z + tolerance)]
        next_layer_coords = np.array(next_layer[['x', 'y', 'z']].values)

        # Check if the distance to the next layer is too large
        distances_to_next_layer = np.linalg.norm(next_layer_coords - pp, axis=1)
        closest_next_point_idx = np.argmin(distances_to_next_layer)
        closest_next_point = next_layer_coords[closest_next_point_idx]

        # Skip the transition if the distance is too large
        if is_jump_in_xy(pp, closest_next_point):
            print(f"Jump detected to next layer at z={next_layer_z}, skipping.")
            continue

        # Update tool position to the closest point and add it to the tool path
        pp = closest_next_point
        ll.append(closest_next_point)

# Save tool path
df_ltp = pd.DataFrame(ll, columns=['x', 'y', 'z'])
df_ltp.to_csv('toolpath.csv', index=False)

# Convert to G-code and save
df_gchord = pd.DataFrame({
    'G': 'G01',
    'x': 'X' + df_ltp['x'].astype(str),
    'y': 'Y' + df_ltp['y'].astype(str),
    'z': 'Z' + df_ltp['z'].astype(str),
    'EOB': ';'
})
df_gchord.to_csv('Gchord.txt', index=False, sep=" ", header=False)


In [21]:
#alpha shape with the updated stl output

import pyvista as pv
import csv
import pandas as pd
import numpy as np
import alphashape
from shapely.geometry import Polygon, MultiPolygon, MultiPoint

pd.options.mode.copy_on_write = True

# Load mesh and prepare CSV file of points
mesh = pv.read('freeform_test_7_pt1.stl')
points = mesh.points
with open('dots.csv', 'w') as f:
    writer = csv.writer(f)
    writer.writerows(points)

df = pd.read_csv('dots.csv').dropna()
df = df.set_axis(['x', 'y', 'z'], axis='columns')

# Remove points close to the bottom plane
df = df[df['z'] > df['z'].min() + 0.1]

# Initial variables
ll = []  # Tool path storage

tolerance = 0.1  # Z tolerance for grouping points into layers
alpha_value = 2  # Alpha shape flexibility
min_layer_distance = 0.1  # Minimum distance between layers to prevent jumps
max_xy_distance = 1  # Maximum allowable distance between points in x and y coordinates
max_z_distance = 0.5  # Maximum Z distance to allow smooth transition between layers

# Function to detect degenerate (collinear or coplanar) points
def is_degenerate(points, tolerance=1e-5):
    if len(points) < 3:
        return True  # Not enough points to form a valid shape
    if np.linalg.matrix_rank(points) < 3:
        return True  # Points are degenerate
    return False

# Function to check if the points form a valid 2D shape (using shapely)
def valid_polygon_2d(points):
    try:
        multipoint = MultiPoint(points)
        hull = multipoint.convex_hull  # Get the convex hull of the points
        if isinstance(hull, Polygon) and len(hull.exterior.coords) >= 3:
            return True  # Valid polygon
        else:
            return False  # Invalid polygon
    except Exception as e:
        return False  # Error case (invalid input)

# # Function to calculate the distance in the x, y plane
# def distance_in_xy(p1, p2):
#     return np.linalg.norm(p1[:2] - p2[:2])  # Only consider x and y distance

# Function to calculate the z distance
def distance_in_z(p1, p2):
    return abs(p1[2] - p2[2])  # Only consider z distance

# Check if the distance to the next layer is too large in both XY and Z dimensions
def is_jump_in_space(current_point, next_point, xy_threshold=max_xy_distance, z_threshold=max_z_distance):
    return (distance_in_xy(current_point, next_point) > xy_threshold) or (distance_in_z(current_point, next_point) > z_threshold)

# Identify the highest layer
initial_layer_z = df['z'].max()
initial_layer = df[df['z'] >= initial_layer_z - tolerance]

# Choose the first point in the highest layer as the starting point
initial_point = initial_layer.iloc[0]
pp = initial_point[['x', 'y', 'z']].values

while len(df) > 0:
    # Identify the highest layer
    current_layer_z = df['z'].max()
    current_layer = df[(df['z'] >= current_layer_z - tolerance) & (df['z'] <= current_layer_z + tolerance)]

    # Skip layers with fewer than 3 points
    if len(current_layer) < 3:
        df = df[df['z'] < current_layer_z - tolerance]
        continue

    # Check if the current layer's points are degenerate (collinear or coplanar)
    if is_degenerate(current_layer[['x', 'y', 'z']].values):
        print(f"Degenerate layer detected at z={current_layer_z}, skipping.")
        df = df[df['z'] < current_layer_z - tolerance]
        continue

    # Extract the 2D points (x, y) for Alpha Shape calculation
    layer_points = current_layer[['x', 'y']].values

    # Check if the 2D points form a valid polygon
    if not valid_polygon_2d(layer_points):
        print(f"Invalid 2D points at z={current_layer_z}, skipping Alpha Shape.")
        df = df[df['z'] < current_layer_z - tolerance]
        continue

    # Filter points using Alpha Shape
    boundary = alphashape.alphashape(layer_points, alpha_value)

    boundary_points_2d = []
    if isinstance(boundary, Polygon):
        boundary_points_2d = np.array(boundary.exterior.coords)
    elif isinstance(boundary, MultiPolygon):
        for poly in boundary.geoms:
            if poly.exterior:
                boundary_points_2d.extend(poly.exterior.coords)
        boundary_points_2d = np.array(boundary_points_2d)
    else:
        boundary_points_2d = layer_points

    # Add z-coordinate back and update the DataFrame
    boundary_points = np.column_stack((boundary_points_2d, np.full(len(boundary_points_2d), current_layer_z)))
    current_layer = pd.DataFrame(boundary_points, columns=['x', 'y', 'z'])

    # Convert to NumPy array for calculations
    current_coords = np.array(current_layer[['x', 'y', 'z']].values)

    # Process points in the current layer with optimized nearest-neighbor sorting
    layer_path = []
    while len(current_coords) > 0:
        distances = np.linalg.norm(current_coords - pp, axis=1)
        closest_idx = np.argmin(distances)
        closest_point = current_coords[closest_idx]

        # Append the closest point and update current position
        layer_path.append(closest_point)
        pp = closest_point

        # Remove the processed point
        current_coords = np.delete(current_coords, closest_idx, axis=0)

    # Close the loop for the current layer
    layer_path.append(layer_path[0])  # Return to the first point
    pp = layer_path[0]  # Update tool position

    # Append the layer path to the overall tool path
    ll.extend(layer_path)

    # Remove processed points from the main DataFrame
    df = df[df['z'] < current_layer_z - tolerance]

    # Process the next layer if it exists
    if len(df) > 0:
        next_layer_z = df['z'].max()
        next_layer = df[(df['z'] >= next_layer_z - tolerance) & (df['z'] <= next_layer_z + tolerance)]
        next_layer_coords = np.array(next_layer[['x', 'y', 'z']].values)

        # Check if the distance to the next layer is too large in XY or Z
        distances_to_next_layer = np.linalg.norm(next_layer_coords - pp, axis=1)
        closest_next_point_idx = np.argmin(distances_to_next_layer)
        closest_next_point = next_layer_coords[closest_next_point_idx]

        # Skip the transition if the jump in XY or Z is too large
        if is_jump_in_space(pp, closest_next_point):
            print(f"Jump detected to next layer at z={next_layer_z}, skipping.")
            continue

        # Update tool position to the closest point and add it to the tool path
        pp = closest_next_point
        ll.append(closest_next_point)

# Save tool path
df_ltp = pd.DataFrame(ll, columns=['x', 'y', 'z'])
df_ltp.to_csv('toolpath.csv', index=False)

# Convert to G-code and save
df_gchord = pd.DataFrame({
    'G': 'G01',
    'x': 'X' + df_ltp['x'].astype(str),
    'y': 'Y' + df_ltp['y'].astype(str),
    'z': 'Z' + df_ltp['z'].astype(str),
    'EOB': ';'
})
df_gchord.to_csv('Gchord.txt', index=False, sep=" ", header=False)


In [24]:
#12/5 Solved the problems with lack of points and increased the layers. (trying to figure out how to maintain the distance between each layer)

import pyvista as pv
import csv
import pandas as pd
import numpy as np
from scipy.spatial import ConvexHull
pd.options.mode.copy_on_write = True


# Load mesh and prepare CSV file of points
mesh = pv.read('freeform_test_7_pt1.stl')
points = mesh.points
with open('dots.csv', 'w') as f:
    writer = csv.writer(f)
    writer.writerows(points)

df = pd.read_csv('dots.csv').dropna()
df = df.set_axis(['x', 'y', 'z'], axis='columns')

# Remove points close to the bottom plane
df = df[df['z'] > df['z'].min() + 0.1]

# Initial variables
ll = []  # Tool path storage
pp = np.array([0, 0, 400])  # Initial tool position
tolerance = 0.33  # Z tolerance for grouping points into layers  

# # Round values for layer selection
# # Round z to the nearest half or whole number
# df['z'] = (df['z'].round() * 3) / 3
# # df['z'] = df['z'].round()
# df['x'] = df['x'].round()
# df['y'] = df['y'].round()

while len(df) > 0:
    # Identify the highest layer
    current_layer_z = df['z'].max()
    current_layer = df[(df['z'] >= current_layer_z - tolerance) & (df['z'] <= current_layer_z + tolerance)]


    # Skip layers with fewer than 3 points
    if len(current_layer) < 3:
        df = df[df['z'] < current_layer_z - tolerance]
        continue

    # Filter points to retain only boundary points using ConvexHull
    layer_points = current_layer[['x', 'y']].values
    hull = ConvexHull(layer_points)
    current_layer = current_layer.iloc[hull.vertices]

    # Convert to NumPy array for calculations
    current_coords = np.array(current_layer[['x', 'y', 'z']].values)

    # Process points in the current layer
    layer_path = []
    while len(current_coords) > 0:
        # Calculate distances from current position to all points in the layer
        distances = np.linalg.norm(current_coords - pp, axis=1)
        closest_idx = np.argmin(distances)
        closest_point = current_coords[closest_idx]

        # Append the closest point and update current position
        layer_path.append(closest_point)
        pp = closest_point

        # Remove the processed point
        current_coords = np.delete(current_coords, closest_idx, axis=0)

    # Close the loop for the current layer
    layer_path.append(layer_path[0])  # Return to the first point
    pp = layer_path[0]  # Update tool position

    # Append the layer path to the overall tool path
    ll.extend(layer_path)

    # Remove processed points from the main DataFrame
    df = df[df['z'] < current_layer_z - tolerance]

    # Find the closest point in the next layer if it exists
    if len(df) > 0:
        next_layer_z = df['z'].max()
        next_layer = df[(df['z'] >= next_layer_z - tolerance) & (df['z'] <= next_layer_z + tolerance)]
        
        # Extract points for the next layer
        next_layer_coords = np.array(next_layer[['x', 'y', 'z']].values)
        
        # Calculate the Convex Hull for the next layer
        if len(next_layer_coords) >= 3:  # Convex Hull requires at least 3 points
            hull = ConvexHull(next_layer_coords[:, :2])  # Use x, y coordinates only
            convex_hull_points = next_layer_coords[hull.vertices]  # Select points on the hull
        else:
            convex_hull_points = next_layer_coords  # If fewer than 3 points, use all points

        # Find the closest point on the Convex Hull to the current position
        distances_to_hull = np.linalg.norm(convex_hull_points - pp, axis=1)
        closest_hull_idx = np.argmin(distances_to_hull)
        closest_point_hull = convex_hull_points[closest_hull_idx]

        # Update the tool position and append the point to the toolpath
        pp = closest_point_hull
        ll.append(closest_point_hull)


# Save tool path
df_ltp = pd.DataFrame(ll, columns=['x', 'y', 'z'])

# # Flip the tool path by reversing the order of the points
# df_ltp = df_ltp.iloc[::-1].reset_index(drop=True)

# Save the flipped tool path
df_ltp.to_csv('toolpath.csv', index=False)

# Convert to G-code and save
df_gchord = pd.DataFrame({
    'G': 'G01',
    'x': 'X' + df_ltp['x'].astype(str),
    'y': 'Y' + df_ltp['y'].astype(str),
    'z': 'Z' + df_ltp['z'].astype(str),
    'EOB': ';'
})
df_gchord.to_csv('Gchord.txt', index=False, sep=" ", header=False)





In [8]:
#12/12 Working on keeping the distance on the bottom layers the same, might need to "expand" the distance or just get rid of those layers all together

'''From here to the next section is setting up the variables for the rest of the code'''

import pyvista as pv
import csv
import pandas as pd
import numpy as np
from scipy.spatial import ConvexHull
from scipy.spatial.distance import pdist, squareform
from scipy.optimize import linear_sum_assignment

pd.options.mode.copy_on_write = True

# Load the STL and converting the dots into a CSV file that python can edit
mesh = pv.read('test_piece_3.stl')
points = mesh.points
with open('dots.csv', 'w') as f:
    writer = csv.writer(f)
    writer.writerows(points)

# Sorting the dots.csv file that was made in the previous step in order to define coordinate points (X,Y,Z) and removing incomplete data (N/A)
df = pd.read_csv('dots.csv').dropna()
df = df.set_axis(['x', 'y', 'z'], axis='columns')

# This is removes the bottom layer of the shape in order to prevent the machine from going back over the already processed points
df = df[df['z'] > df['z'].min() + 0.1]

# Setting variables for the rest of the code, specifying the number of layers wanted 
num_layers = 175  # Specify the desired number of layers

# The computatoin for ensuring that the Z-coordinates or "layers" have the same spacing between each other
z_min, z_max = df['z'].min(), df['z'].max()
layer_spacing = (z_max - z_min) / num_layers

# This groups the Z coordinates so that the layers are defined
df['z'] = np.round((df['z'] - z_min) / layer_spacing) * layer_spacing + z_min



# This section is used to sort the points using TSP to ensure that there is no long jumps in the tool path and that the tool path stays 'clean'. This is just used to define a custom function.
def solve_tsp(points):
    # Compute pairwise distances
    distance_matrix = squareform(pdist(points))
    # Solve TSP using the Hungarian algorithm
    row_ind, col_ind = linear_sum_assignment(distance_matrix)
    return col_ind

# This section is used to set the points to be processed in the same way for every layer ensuring that the tool path stays 'clean'. This is just used to define a custom function.
def sort_clockwise(points):
    center = points.mean(axis=0)  # Find centroid of the points
    angles = np.arctan2(points[:, 1] - center[1], points[:, 0] - center[0])  # Calculate angles
    return points[np.argsort(angles)]  # Sort points by angle

'''Creating the Tool path for the machine to process'''

# Initialize variables
ll = []  # This stores the tool path so that it can be saved to a txt file later
pp = np.array([0, 0, 400])  # This defines a starting point for the program to start processing from, this can be adjusted but changing this shouldnt matter to how the tool path looks.

while len(df) > 0:
    # This identifys the current layer and identifies the greatest Z coordinate or highest layer.
    current_layer_z = df['z'].max()
    current_layer = df[df['z'] == current_layer_z]

    # This layer skips any layers with fewer than 3 points because ConvexHull is unable to process layers with less than 3 points. This stops the program from breaking.
    if len(current_layer) < 3:
        df = df[df['z'] < current_layer_z]
        continue

    # Filter points to retain only boundary points using ConvexHull. This ensures that only the outermost points are processed and saved to the tool path.
    layer_points = current_layer[['x', 'y']].values
    hull = ConvexHull(layer_points)
    hull_points = current_layer.iloc[hull.vertices]

    # The next three lines are for using TSP to process the hull points
    hull_coords = hull_points[['x', 'y']].values
    tsp_order = solve_tsp(hull_coords)
    tsp_sorted_points = hull_coords[tsp_order]

    # This line is used to ensure that the hull points are sorted and processed in a clockwise direction.
    sorted_hull_points = sort_clockwise(tsp_sorted_points)

    # Rebuild the sorted layer's coordinates to add the Z coordinate becase ConvexHull only sorts the X and Y coordinates
    current_coords = np.hstack([
        sorted_hull_points,  # X and Y coordinates
        np.full((sorted_hull_points.shape[0], 1), current_layer_z)  # Z coordinate
    ])

    # Process points in the current layer
    layer_path = list(current_coords)

    # This part of the code makes the tool path return to the first point of the layer and then adds this movement to the end of the tool path.
    layer_path.append(layer_path[0])  # Return to the first point
    pp = layer_path[0]  # Update tool position
    ll.extend(layer_path)

    # Remove processed points from the main DataFrame then from here it repeats untill all of the layers are processed
    df = df[df['z'] < current_layer_z]


'''Bellow is converting the tool path into G-Code so that it can be used for actual machining'''
# Save tool path to a CSV file to convert the tool path to G-code
df_ltp = pd.DataFrame(ll, columns=['x', 'y', 'z'])
df_ltp.to_csv('toolpath.csv', index=False)

# Convert to G-code and save as a text file. (Can be changed to CSV or any file type.)
df_gchord = pd.DataFrame({
    'G': 'G01',
    'x': 'X' + df_ltp['x'].astype(str),
    'y': 'Y' + df_ltp['y'].astype(str),
    'z': 'Z' + df_ltp['z'].astype(str),
    'EOB': ';'
})
df_gchord.to_csv('Gchord.txt', index=False, sep=" ", header=False)


In [14]:
# 12/19 Adding general Gcode parameters so that the code actually runs in the CNC machine

'''From here to the next section is setting up the variables for the rest of the code'''
# Importing libraries that will be used for the rest of the code
import pyvista as pv
import csv
import pandas as pd
import numpy as np
from scipy.spatial import ConvexHull
from scipy.spatial.distance import pdist, squareform
from scipy.optimize import linear_sum_assignment

# Load the STL and converting the dots into a CSV file that python can edit
mesh = pv.read('freeform_test_9.stl')
points = mesh.points
pd.options.mode.copy_on_write = True

# Sorting the dots.csv file that was made in the previous step in order to define coordinate points (X,Y,Z)
with open('dots.csv', 'w') as f:
    csv.writer(f).writerows(points)
df = pd.read_csv('dots.csv', header=None, names=['x', 'y', 'z']).dropna()

# This is removes the bottom layer of the shape in order to prevent the machine from going back over the already processed points
df = df[df['z'] > df['z'].min() + 0.001]

# Specifying the number of layers wanted; adjust until your toolpath is clean and does not have large jumps to other parts of the workpiece 
num_layers = 100  

# The computatoin for ensuring that the Z-coordinates or "layers" have the same spacing between each other
z_min, z_max = df['z'].min(), df['z'].max()
layer_spacing = (z_max - z_min) / num_layers


# This groups the Z coordinates so that the layers are defined
df['z'] = np.round((df['z'] - z_min) / layer_spacing) * layer_spacing + z_min

'''Modules for simplifying the tool path process'''
# Module for calculating the circular shapes
def is_circular_shape(points, tolerance=0.1):  # The tolerance must be adjusted according to the dimensions of the model, the bigger the number the less amount of points are selected
    center = points.mean(axis=0)
    distances = np.linalg.norm(points - center, axis=1)
    return np.abs(distances - distances.mean()).max() < tolerance

# Module for calculating TSP
def solve_tsp(points): 
    distance_matrix = squareform(pdist(points))
    _, col_ind = linear_sum_assignment(distance_matrix)
    return col_ind

# Module for sorting the points in a clockwise direction
def sort_clockwise(points):  
    center = points.mean(axis=0)
    angles = np.arctan2(points[:, 1] - center[1], points[:, 0] - center[0])
    return points[np.argsort(angles)]

# Module for processing circular layers
def process_circular_layer(layer_points):  
    return sort_clockwise(layer_points)

# Module for processing non-circular layers
def process_non_circular_layer(layer_points): 
    tsp_order = solve_tsp(layer_points)
    return sort_clockwise(layer_points[tsp_order])

'''Creating the Tool path for the machine to process'''

# Initialize toolpath
ll = [] # This stores the tool path so that it can be saved to a txt file later
while len(df) > 0:
    # This identifys the current layer and identifies the greatest Z coordinate or highest layer.
    current_layer_z = df['z'].max()
    current_layer = df[df['z'] == current_layer_z]

    # This layer skips any layers with fewer than 3 points because ConvexHull is unable to process layers with less than 3 points. This stops the program from breaking.
    if len(current_layer) < 3:
        df = df[df['z'] < current_layer_z]
        continue

    # Filter points to retain only boundary points using ConvexHull. This ensures that only the outermost points are processed and saved to the tool path.
    layer_points = current_layer[['x', 'y']].values
    hull = ConvexHull(layer_points)
    hull_coords = layer_points[hull.vertices]

    # Determine if the shape is circular or non-circular
    if is_circular_shape(hull_coords):
        sorted_points = process_circular_layer(hull_coords)
    else:
        sorted_points = process_non_circular_layer(hull_coords)

    # Add Z-coordinate to sorted points
    sorted_layer = np.hstack([sorted_points, np.full((sorted_points.shape[0], 1), current_layer_z)])

    # This part of the code makes the tool path return to the first point of the layer and then adds this movement to the end of the tool path.
    layer_path = list(sorted_layer)
    layer_path.append(layer_path[0])  # Return to the first point
    ll.extend(layer_path)

    # Remove processed points from the main DataFrame then from here it repeats untill all of the layers are processed
    df = df[df['z'] < current_layer_z]

# Save toolpath to a CSV
ll = np.round(ll, 3)
pd.DataFrame(ll, columns=['x', 'y', 'z']).to_csv('toolpath.csv', index=False)

'''Bellow is converting the tool path into G-Code and adding peramiters to the Gcode so that the tool will actually move during machining'''

# Convert to G-code and save as a text file. (Can be changed to CSV or any file type.)
df_ltp = pd.read_csv('toolpath.csv')
df_gchord = pd.DataFrame({
    'G': 'G01',
    'x': 'X' + df_ltp['x'].astype(str),
    'y': 'Y' + df_ltp['y'].astype(str),
    'z': 'Z' + df_ltp['z'].astype(str),
    'EOB': ';'
})

# Define the starting and ending G-code lines, The coordinates for the following sections (gcode_header and gcode_footer) can be changed depending on the size of the work area for the CNC machine
gcode_header = [
    "G90 ; (Absolute positioning)",
    "G21 ; (Set units to millimeters)",
    "G92 X0.0 Y0.0 Z0 ; (Set workplace origin)",
    "F1000 ; (Set feedrate)",
    "G01 X0.0 Y0.0 Z20 ; (Setting ensuring that the tool will move down to the work area)"
]
# Note that for the line "G01 X0.0 Y0.0 Z20" and "G01 Z20" the Z coordinate MUST be changed depending on the work areas height so that the tool goes down instead of going up from the work area origin

gcode_footer = [
    "G01 Z20 ; (Move straight up to the origin height)",
    "M30 ; (End of program)"
]  

# Add the header, G-code, and footer to the file
with open('Gchord.txt', 'w') as f:
    # Write the header
    for line in gcode_header:
        f.write(line + '\n')
    
    # Write the G-code toolpath
    df_gchord.to_csv(f, index=False, sep=" ", header=False, mode='a')
    
    # Write the footer
    for line in gcode_footer:
        f.write(line + '\n')


In [6]:
# 1/9 adjusting for to make sure that the distance between the z coordinate is uniform to improve the strength of the material after incrmental forming

'''From here to the next section is setting up the variables for the rest of the code'''
# Importing libraries that will be used for the rest of the code
import pyvista as pv
import csv
import pandas as pd
import numpy as np
from scipy.spatial import ConvexHull
from scipy.spatial.distance import pdist, squareform
from scipy.optimize import linear_sum_assignment

# Load the STL and converting the dots into a CSV file that python can edit
mesh = pv.read('freeform_test_11.stl')
points = mesh.points
pd.options.mode.copy_on_write = True

# Sorting the dots.csv file that was made in the previous step in order to define coordinate points (X,Y,Z)
with open('dots.csv', 'w') as f:
    csv.writer(f).writerows(points)
df = pd.read_csv('dots.csv', header=None, names=['x', 'y', 'z']).dropna()

# This is removes the bottom layer of the shape in order to prevent the machine from going back over the already processed points
df = df[df['z'] > df['z'].min() + 0.01]

# Specifying the number of layers wanted; adjust until your toolpath is clean and does not have large jumps to other parts of the workpiece 
num_layers = 100

# Specify the desired layer spacing as a variable
layer_spacing = 0.1  # Set the distance between layers (in your desired units)

def enforce_layer_spacing(df, layer_spacing):
    """Ensure layers are spaced with a fixed distance defined by layer_spacing."""
    z_min = df['z'].min()
    # Compute possible layer positions based on layer spacing
    layers = np.arange(z_min, df['z'].max() + layer_spacing, layer_spacing)
    
    # Assign each point to the closest layer
    df['z'] = layers[np.argmin(np.abs(df['z'].values[:, None] - layers), axis=1)]
    return df

# Apply the function to enforce fixed layer spacing
df = enforce_layer_spacing(df, layer_spacing)

'''Modules for simplifying the tool path process'''
# Module for calculating the circular shapes
def is_circular_shape(points, tolerance=0.01):  # The tolerance must be adjusted according to the dimensions of the model, the bigger the number the less amount of points are selected
    center = points.mean(axis=0)
    distances = np.linalg.norm(points - center, axis=1)
    return np.abs(distances - distances.mean()).max() < tolerance

# Module for calculating TSP
def solve_tsp(points): 
    distance_matrix = squareform(pdist(points))
    _, col_ind = linear_sum_assignment(distance_matrix)
    return col_ind

# Module for sorting the points in a clockwise direction
def sort_clockwise(points):  
    center = points.mean(axis=0)
    angles = np.arctan2(points[:, 1] - center[1], points[:, 0] - center[0])
    return points[np.argsort(angles)]

# Module for processing circular layers
def process_circular_layer(layer_points):  
    return sort_clockwise(layer_points)

# Module for processing non-circular layers
def process_non_circular_layer(layer_points): 
    tsp_order = solve_tsp(layer_points)
    return sort_clockwise(layer_points[tsp_order])

'''Creating the Tool path for the machine to process'''

# Initialize toolpath
ll = [] # This stores the tool path so that it can be saved to a txt file later
while len(df) > 0:
    # This identifys the current layer and identifies the greatest Z coordinate or highest layer.
    current_layer_z = df['z'].max()
    current_layer = df[df['z'] == current_layer_z]

    # This layer skips any layers with fewer than 3 points because ConvexHull is unable to process layers with less than 3 points. This stops the program from breaking.
    if len(current_layer) < 3:
        df = df[df['z'] < current_layer_z]
        continue

    # Filter points to retain only boundary points using ConvexHull. This ensures that only the outermost points are processed and saved to the tool path.
    layer_points = current_layer[['x', 'y']].values
    hull = ConvexHull(layer_points)
    hull_coords = layer_points[hull.vertices]

    # Determine if the shape is circular or non-circular
    if is_circular_shape(hull_coords):
        sorted_points = process_circular_layer(hull_coords)
    else:
        sorted_points = process_non_circular_layer(hull_coords)

    # Add Z-coordinate to sorted points
    sorted_layer = np.hstack([sorted_points, np.full((sorted_points.shape[0], 1), current_layer_z)])

    # This part of the code makes the tool path return to the first point of the layer and then adds this movement to the end of the tool path.
    layer_path = list(sorted_layer)
    layer_path.append(layer_path[0])  # Return to the first point
    ll.extend(layer_path)

    # Remove processed points from the main DataFrame then from here it repeats untill all of the layers are processed
    df = df[df['z'] < current_layer_z]

    unique_z = np.sort(df['z'].unique())
    print("Layer spacings:", np.diff(unique_z))

# Save toolpath to a CSV
ll = np.round(ll, 3)
pd.DataFrame(ll, columns=['x', 'y', 'z']).to_csv('toolpath.csv', index=False)

'''Bellow is converting the tool path into G-Code and adding peramiters to the Gcode so that the tool will actually move during machining'''

# Convert to G-code and save as a text file. (Can be changed to CSV or any file type.)
df_ltp = pd.read_csv('toolpath.csv')
df_gchord = pd.DataFrame({
    'G': 'G01',
    'x': 'X' + df_ltp['x'].astype(str),
    'y': 'Y' + df_ltp['y'].astype(str),
    'z': 'Z' + df_ltp['z'].astype(str),
    'EOB': ';'
})

# Define the starting and ending G-code lines, The coordinates for the following sections (gcode_header and gcode_footer) can be changed depending on the size of the work area for the CNC machine
gcode_header = [
    "G90 ; (Absolute positioning)",
    "G21 ; (Set units to millimeters)",
    "G92 X0.0 Y0.0 Z0 ; (Set workplace origin)",
    "F1000 ; (Set feedrate)",
    "G01 X0.0 Y0.0 Z20 ; (Setting ensuring that the tool will move down to the work area)"
]
# Note that for the line "G01 X0.0 Y0.0 Z20" and "G01 Z20" the Z coordinate MUST be changed depending on the work areas height so that the tool goes down instead of going up from the work area origin

gcode_footer = [
    "G01 Z20 ; (Move straight up to the origin height)",
    "M30 ; (End of program)"
]  

# Add the header, G-code, and footer to the file
with open('Gchord.txt', 'w') as f:
    # Write the header
    for line in gcode_header:
        f.write(line + '\n')
    
    # Write the G-code toolpath
    df_gchord.to_csv(f, index=False, sep=" ", header=False, mode='a')
    
    # Write the footer
    for line in gcode_footer:
        f.write(line + '\n')

Layer spacings: [0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1
 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1
 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1
 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1
 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1
 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1]
Layer spacings: [0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1
 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1
 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1
 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1
 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1
 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1]
Layer spacings: [0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1
 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1
 0.1

In [29]:
# 4/14 
'''From here to the next section is setting up the variables for the rest of the code'''
# Importing libraries that will be used for the rest of the code
import pyvista as pv
import csv
import pandas as pd
import numpy as np
from scipy.spatial import ConvexHull
from scipy.spatial.distance import pdist, squareform
from scipy.optimize import linear_sum_assignment

# Load the STL and converting the points into mesh
mesh = pv.read('freeform_test_17v3.stl')
mesh = mesh.subdivide(2)  # Increase this number (e.g., 3 or 4) for more points
points = mesh.points

pd.options.mode.copy_on_write = True

# Save mesh points to CSV so that the program can create the toolpath (this is all raw data)
with open('dots.csv', 'w') as f:
    csv.writer(f).writerows(points)
df = pd.read_csv('dots.csv', header=None, names=['x', 'y', 'z']).dropna()

# This is removes the bottom layer of the shape in order to prevent the machine from going back over the already processed points
df = df[df['z'] > df['z'].min() + 0.1]

# Specify the desired layer spacing as a variable
layer_spacing = 0.2  # Set the distance between layers (in your desired units)

'''Modules for simplifying the tool path process'''
# Module for calculating the circular shapes

def enforce_layer_spacing(df, layer_spacing):
    """Ensure layers are spaced with a fixed distance defined by layer_spacing."""
    z_min = df['z'].min()
    layers = np.arange(z_min, df['z'].max() + layer_spacing, layer_spacing) # Compute possible layer positions based on layer spacing
    df['z'] = layers[np.argmin(np.abs(df['z'].values[:, None] - layers), axis=1)] # Assign each point to the closest layer
    return df

df = enforce_layer_spacing(df, layer_spacing)

def is_circular_shape(points, tolerance=0.1): # The tolerance must be adjusted according to the dimensions of the model, the bigger the number the less amount of points are selected
    center = points.mean(axis=0)
    distances = np.linalg.norm(points - center, axis=1)
    return np.abs(distances - distances.mean()).max() < tolerance


# Module for calculating TSP
def solve_tsp(points):
    distance_matrix = squareform(pdist(points))
    _, col_ind = linear_sum_assignment(distance_matrix)
    return col_ind


# Module for sorting the points in a clockwise direction
def sort_clockwise(points):
    center = points.mean(axis=0)
    angles = np.arctan2(points[:, 1] - center[1], points[:, 0] - center[0])
    return points[np.argsort(angles)]


# Module for processing circular layers
def process_circular_layer(layer_points):
    return sort_clockwise(layer_points)


# Module for processing non-circular layers
def process_non_circular_layer(layer_points):
    tsp_order = solve_tsp(layer_points)
    return sort_clockwise(layer_points[tsp_order])

'''Creating the Tool path for the machine to process'''

# Initialize toolpath
ll = []
prev_exit_point = None

while len(df) > 0:
    # This identifys the current layer and identifies the greatest Z coordinate or highest layer.
    current_layer_z = df['z'].max()
    current_layer = df[df['z'] == current_layer_z]

    # This layer skips any layers with fewer than 3 points because ConvexHull is unable to process layers with less than 3 points. This stops the program from breaking.
    if len(current_layer) < 3:
        df = df[df['z'] < current_layer_z]
        continue
    
    # Filter points to retain only boundary points using ConvexHull. This ensures that only the outermost points are processed and saved to the tool path.
    layer_points = current_layer[['x', 'y']].values
    hull = ConvexHull(layer_points)
    hull_coords = layer_points[hull.vertices]

    # Determine if the shape is circular or non-circular
    if is_circular_shape(hull_coords):
        sorted_points = process_circular_layer(hull_coords)
    else:
        sorted_points = process_non_circular_layer(hull_coords)

    # Closest point from previous layer to start
    if prev_exit_point is not None:
        dists = np.linalg.norm(sorted_points - prev_exit_point[:2], axis=1)
        start_idx = np.argmin(dists)
        sorted_points = np.roll(sorted_points, -start_idx, axis=0)
    
    # Add Z-coordinate to sorted points
    sorted_layer = np.hstack([sorted_points, np.full((sorted_points.shape[0], 1), current_layer_z)])
    
    # This part of the code makes the tool path return to the first point of the layer and then adds this movement to the end of the tool path.
    layer_path = list(sorted_layer)
    layer_path.append(layer_path[0])  # Return to the first point
    ll.extend(layer_path)

    prev_exit_point = layer_path[0]  # For continuous connection
    
    # Remove processed points from the main DataFrame then from here it repeats untill all of the layers are processed
    df = df[df['z'] < current_layer_z]

    #prints the layer spacings so that we can ensure that the layers are the same distance apart from each other
    unique_z = np.sort(df['z'].unique())
    print("Layer spacings:", np.diff(unique_z))

# Save toolpath to a CSV
ll = np.round(ll, 3)
pd.DataFrame(ll, columns=['x', 'y', 'z']).to_csv('toolpath.csv', index=False)

'''Bellow is converting the tool path into G-Code and adding peramiters to the Gcode so that the tool will actually move during machining'''
# Convert to G-code and save as a text file. (Can be changed to CSV or any file type.)
df_ltp = pd.read_csv('toolpath.csv')
df_gchord = pd.DataFrame({
    'G': 'G01',
    'x': 'X' + df_ltp['x'].astype(str),
    'y': 'Y' + df_ltp['y'].astype(str),
    'z': 'Z' + df_ltp['z'].astype(str),
    'EOB': ';'
})

# Define the starting and ending G-code lines.
# The coordinates for the following sections (gcode_header and gcode_footer) can be changed depending on the size of the work area for the CNC machine
gcode_header = [
    "G90 ; (Absolute positioning)",
    "G21 ; (Set units to millimeters)",
    "G92 X0.0 Y0.0 Z0 ; (Set workplace origin)",
    "F1000 ; (Set feedrate)",
    "G01 X0.0 Y0.0 Z20 ; (Move to safe height)"
]
# Note that for the line "G01 X0.0 Y0.0 Z20" and "G01 Z20" the Z coordinate MUST be changed depending on the work areas height.
# So that the tool goes down instead of going up from the work area origin

gcode_footer = [
    "G01 Z20 ; (Retract tool)",
    "M30 ; (End of program)"
]

# Add the header, G-code, and footer to the file
with open('Gchord.txt', 'w') as f:
    # Write the header
    for line in gcode_header:
        f.write(line + '\n')
    
    # Write the G-code toolpath
    df_gchord.to_csv(f, index=False, sep=" ", header=False, mode='a')
    
    # Write the footer
    for line in gcode_footer:
        f.write(line + '\n')

Layer spacings: [0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2
 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2
 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2
 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2
 0.2]
Layer spacings: [0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2
 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2
 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2
 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2]
Layer spacings: [0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2
 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2
 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2
 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2]
Layer spacings: [0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2

In [42]:
# 5/8 fixes to my code
'''From here to the next section is setting up the variables for the rest of the code'''
# Importing libraries that will be used for the rest of the code
import pyvista as pv
import csv
import pandas as pd
import numpy as np
from scipy.spatial import ConvexHull
from scipy.spatial.distance import pdist, squareform
from scipy.optimize import linear_sum_assignment

# Load the STL and converting the points into mesh
mesh = pv.read('freeform_test_17.stl')
mesh = mesh.subdivide(2)  # Increase this number (e.g., 3 or 4) for more points
points = mesh.points

pd.options.mode.copy_on_write = True

# Save mesh points to CSV so that the program can create the toolpath (this is all raw data)
with open('dots.csv', 'w') as f:
    csv.writer(f).writerows(points)
df = pd.read_csv('dots.csv', header=None, names=['x', 'y', 'z']).dropna()

# This is removes the bottom layer of the shape in order to prevent the machine from going back over the already processed points
df = df[df['z'] > df['z'].min() + 0.1]

# Specify the desired layer spacing as a variable
layer_spacing = 0.2  # Set the distance between layers (in your desired units)

'''Modules for simplifying the tool path process'''
# Module for calculating the circular shapes

def enforce_layer_spacing(df, layer_spacing):
    """Ensure layers are spaced with a fixed distance defined by layer_spacing."""
    z_min = df['z'].min()
    layers = np.arange(z_min, df['z'].max() + layer_spacing, layer_spacing) # Compute possible layer positions based on layer spacing
    df['z'] = layers[np.argmin(np.abs(df['z'].values[:, None] - layers), axis=1)] # Assign each point to the closest layer
    return df

df = enforce_layer_spacing(df, layer_spacing)

def is_circular_shape(points, tolerance=0.1): # The tolerance must be adjusted according to the dimensions of the model, the bigger the number the less amount of points are selected
    center = points.mean(axis=0)
    distances = np.linalg.norm(points - center, axis=1)
    return np.abs(distances - distances.mean()).max() < tolerance


# Module for calculating TSP
def solve_tsp(points):
    distance_matrix = squareform(pdist(points))
    _, col_ind = linear_sum_assignment(distance_matrix)
    return col_ind


# Module for sorting the points in a clockwise direction
def sort_clockwise(points):
    center = points.mean(axis=0)
    angles = np.arctan2(points[:, 1] - center[1], points[:, 0] - center[0])
    return points[np.argsort(angles)]


# Module for processing circular layers
def process_circular_layer(layer_points):
    return sort_clockwise(layer_points)


# Module for processing non-circular layers
def process_non_circular_layer(layer_points):
    tsp_order = solve_tsp(layer_points)
    return sort_clockwise(layer_points[tsp_order])

'''Creating the Tool path for the machine to process'''

# Initialize toolpath
ll = []
prev_exit_point = None

while len(df) > 0:
    # This identifys the current layer and identifies the greatest Z coordinate or highest layer.
    current_layer_z = df['z'].max()
    current_layer = df[df['z'] == current_layer_z]

    # This layer skips any layers with fewer than 3 points because ConvexHull is unable to process layers with less than 3 points. This stops the program from breaking.
    if len(current_layer) < 3:
        df = df[df['z'] < current_layer_z]
        continue
    
    # Filter points to retain only boundary points using ConvexHull. This ensures that only the outermost points are processed and saved to the tool path.
    layer_points = current_layer[['x', 'y']].values
    hull = ConvexHull(layer_points)
    hull_coords = layer_points[hull.vertices]

    # Determine if the shape is circular or non-circular
    if is_circular_shape(hull_coords):
        sorted_points = process_circular_layer(hull_coords)
    else:
        sorted_points = process_non_circular_layer(hull_coords)

    # Closest point from previous layer to start
    if prev_exit_point is not None:
        dists = np.linalg.norm(sorted_points - prev_exit_point[:2], axis=1)
        start_idx = np.argmin(dists)
        sorted_points = np.roll(sorted_points, -start_idx, axis=0)
    
    # Add Z-coordinate to sorted points
    sorted_layer = np.hstack([sorted_points, np.full((sorted_points.shape[0], 1), current_layer_z)])
    
    # This part of the code makes the tool path return to the first point of the layer and then adds this movement to the end of the tool path.
    layer_path = list(sorted_layer)
    layer_path.append(layer_path[0])  # Return to the first point
    ll.extend(layer_path)

    prev_exit_point = layer_path[0]  # For continuous connection
    
    # Remove processed points from the main DataFrame then from here it repeats untill all of the layers are processed
    df = df[df['z'] < current_layer_z]

    #prints the layer spacings so that we can ensure that the layers are the same distance apart from each other
    unique_z = np.sort(df['z'].unique())
    print("Layer spacings:", np.diff(unique_z))

# Save toolpath to a CSV
ll = np.round(ll, 3)
pd.DataFrame(ll, columns=['x', 'y', 'z']).to_csv('toolpath.csv', index=False)

'''Bellow is converting the tool path into G-Code and adding peramiters to the Gcode so that the tool will actually move during machining'''
# Convert to G-code and save as a text file. (Can be changed to CSV or any file type.)
df_ltp = pd.read_csv('toolpath.csv')
df_gchord = pd.DataFrame({
    'G': 'G01',
    'x': 'X' + df_ltp['x'].astype(str),
    'y': 'Y' + df_ltp['y'].astype(str),
    'z': 'Z' + df_ltp['z'].astype(str),
    'EOB': ';'
})

# Define the starting and ending G-code lines.
# The coordinates for the following sections (gcode_header and gcode_footer) can be changed depending on the size of the work area for the CNC machine

# Dynamically move above the first point at a safe height
first_x = round(df_ltp.iloc[0]['x'], 3)
first_y = round(df_ltp.iloc[0]['y'], 3)
safe_z = 50  # Set your safe Z height here

gcode_header = [
    "G90 ; (Absolute positioning)",
    "G21 ; (Set units to millimeters)",
    # "G92 X0.0 Y0.0 Z0 ; (Set workplace origin)",
    "S500 (setting spindle speed)",
    "F1000 ; (Set feedrate)",
    f"G01 X{first_x} Y{first_y} Z0.0 ; (Move 50 mm above first point)",
    f"G01 Z{df_ltp.iloc[0]['z']} ; (Lower to first point Z)"
]

# Move straight up from the last XY to safe Z height
last_x = round(df_ltp.iloc[-1]['x'], 3)
last_y = round(df_ltp.iloc[-1]['y'], 3)

gcode_footer = [
    f"G01 X{last_x} Y{last_y} Z{safe_z} ; (Retract tool straight up from last point)",
    "M30 ; (End of program)"
]

# Add the header, G-code, and footer to the file
with open('Gchord.txt', 'w') as f:
    # Write the header
    for line in gcode_header:
        f.write(line + '\n')
    
    # Write the G-code toolpath
    df_gchord.to_csv(f, index=False, sep=" ", header=False, mode='a')
    
    # Write the footer
    for line in gcode_footer:
        f.write(line + '\n')


Layer spacings: [0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2
 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2
 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2
 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2
 0.2]
Layer spacings: [0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2
 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2
 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2
 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2]
Layer spacings: [0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2
 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2
 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2
 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2]
Layer spacings: [0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2

In [39]:
# 5/14 fixes to my code 
'''From here to the next section is setting up the variables for the rest of the code'''
# Importing libraries that will be used for the rest of the code
import pyvista as pv
import csv
import pandas as pd
import numpy as np
from scipy.spatial import ConvexHull
from scipy.spatial.distance import pdist, squareform
from scipy.optimize import linear_sum_assignment

# Load the STL and converting the points into mesh
mesh = pv.read('freeform_test_17v3.stl')
mesh = mesh.subdivide(2)  # Increase this number (e.g., 3 or 4) for more points
points = mesh.points

pd.options.mode.copy_on_write = True

# Save mesh points to CSV so that the program can create the toolpath (this is all raw data)
with open('dots.csv', 'w') as f:
    csv.writer(f).writerows(points)
df = pd.read_csv('dots.csv', header=None, names=['x', 'y', 'z']).dropna()

# This is removes the bottom layer of the shape in order to prevent the machine from going back over the already processed points
df = df[df['z'] > df['z'].min() + 0.1]

# Specify the desired layer spacing as a variable
layer_spacing = 0.2  # Set the distance between layers (in your desired units)

'''Modules for simplifying the tool path process'''
# Module for calculating the circular shapes

def enforce_layer_spacing(df, layer_spacing):
    """Ensure layers are spaced with a fixed distance defined by layer_spacing."""
    z_min = df['z'].min()
    layers = np.arange(z_min, df['z'].max() + layer_spacing, layer_spacing) # Compute possible layer positions based on layer spacing
    df['z'] = layers[np.argmin(np.abs(df['z'].values[:, None] - layers), axis=1)] # Assign each point to the closest layer
    return df

df = enforce_layer_spacing(df, layer_spacing)

def is_circular_shape(points, tolerance=0.1): # The tolerance must be adjusted according to the dimensions of the model, the bigger the number the less amount of points are selected
    center = points.mean(axis=0)
    distances = np.linalg.norm(points - center, axis=1)
    return np.abs(distances - distances.mean()).max() < tolerance


# Module for calculating TSP
def solve_tsp(points):
    distance_matrix = squareform(pdist(points))
    _, col_ind = linear_sum_assignment(distance_matrix)
    return col_ind


# Module for sorting the points in a clockwise direction
def sort_clockwise(points):
    center = points.mean(axis=0)
    angles = np.arctan2(points[:, 1] - center[1], points[:, 0] - center[0])
    return points[np.argsort(angles)]


# Module for processing circular layers
def process_circular_layer(layer_points):
    return sort_clockwise(layer_points)


# Module for processing non-circular layers
def process_non_circular_layer(layer_points):
    tsp_order = solve_tsp(layer_points)
    return sort_clockwise(layer_points[tsp_order])

'''Creating the Tool path for the machine to process'''

# Initialize toolpath
ll = []
prev_exit_point = None
layer_index = 0  # Layer counter to optionally alternate or track layers

while len(df) > 0:
    # This identifies the current layer by locating the Z with the largest absolute value
    current_layer_z = df.loc[df['z'].abs().idxmax(), 'z']
    current_layer = df[df['z'] == current_layer_z]

    # This layer skips any layers with fewer than 3 points because ConvexHull is unable to process layers with less than 3 points. This stops the program from breaking.
    if len(current_layer) < 3:
        df = df[df['z'] != current_layer_z]
        continue

    # Filter points to retain only boundary points using ConvexHull. This ensures that only the outermost points are processed and saved to the tool path.
    layer_points = current_layer[['x', 'y']].values
    hull = ConvexHull(layer_points)
    hull_coords = layer_points[hull.vertices]

    # Determine if the shape is circular or non-circular
    if is_circular_shape(hull_coords):
        sorted_points = process_circular_layer(hull_coords)
    else:
        sorted_points = process_non_circular_layer(hull_coords)

    # Closest point from previous layer to start
    if prev_exit_point is not None:
        dists = np.linalg.norm(sorted_points - prev_exit_point[:2], axis=1)
        start_idx = np.argmin(dists)
        sorted_points = np.roll(sorted_points, -start_idx, axis=0)

    # Add Z-coordinate to sorted points
    sorted_layer = np.hstack([sorted_points, np.full((sorted_points.shape[0], 1), current_layer_z)])

    # This part of the code makes the tool path return to the first point of the layer
    layer_path = list(sorted_layer)

    # 🔄 Reverse the layer path direction
    layer_path = layer_path[::-1]

    # Then return to the (new) first point and add this movement to the end of the tool path
    layer_path.append(layer_path[0])  # Return to the first point
    ll.extend(layer_path)

    prev_exit_point = layer_path[0]  # For continuous connection

    # Remove processed points from the main DataFrame then from here it repeats until all of the layers are processed
    df = df[df['z'] != current_layer_z]

    # Prints the layer spacings so that we can ensure that the layers are the same distance apart from each other
    unique_z = np.sort(df['z'].unique())
    print("Layer spacings:", np.diff(unique_z))

    layer_index += 1  # Optional use for alternating logic

# Save toolpath to a CSV
ll = np.round(ll, 3)
pd.DataFrame(ll, columns=['x', 'y', 'z']).to_csv('toolpath.csv', index=False)

'''Below is converting the tool path into G-Code and adding parameters to the G-code so that the tool will actually move during machining'''
# Convert to G-code and save as a text file. (Can be changed to CSV or any file type.)
df_ltp = pd.read_csv('toolpath.csv')
df_gchord = pd.DataFrame({
    'G': 'G01',
    'x': 'X' + df_ltp['x'].astype(str),
    'y': 'Y' + df_ltp['y'].astype(str),
    'z': 'Z' + df_ltp['z'].astype(str),
    'EOB': ';'
})

# Define the starting and ending G-code lines.
# The coordinates for the following sections (gcode_header and gcode_footer) can be changed depending on the size of the work area for the CNC machine

# Dynamically move above the first point at a safe height
first_x = round(df_ltp.iloc[0]['x'], 3)
first_y = round(df_ltp.iloc[0]['y'], 3)
safe_z = 50  # Set your safe Z height here
start_z = -50  # Drop 50 mm below origin

gcode_header = [
    "G90 ; (Absolute positioning)",
    "G21 ; (Set units to millimeters)",
    "G92 X0.0 Y0.0 Z0 ; (Set workplace origin)",
    "S500 (setting spindle speed)",
    "F1000 ; (Set feedrate)",
    f"G01 X0 Y0 Z{start_z} ; (Lower 50 mm from origin)",
    f"G01 X{first_x} Y{first_y} Z{df_ltp.iloc[0]['z']} ; (Move to start point)"
]

# Move straight up from the last XY to safe Z height
last_x = round(df_ltp.iloc[-1]['x'], 3)
last_y = round(df_ltp.iloc[-1]['y'], 3)

gcode_footer = [
    f"G01 X{last_x} Y{last_y} Z{safe_z} ; (Retract tool straight up from last point)",
    "M30 ; (End of program)"
]

# Add the header, G-code, and footer to the file
with open('Gchord.txt', 'w') as f:
    # Write the header
    for line in gcode_header:
        f.write(line + '\n')

    # Write the G-code toolpath
    df_gchord.to_csv(f, index=False, sep=" ", header=False, mode='a')

    # Write the footer
    for line in gcode_footer:
        f.write(line + '\n')


Layer spacings: [0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2
 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2
 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2
 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2
 0.2]
Layer spacings: [0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2
 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2
 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2
 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2]
Layer spacings: [0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2
 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2
 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2
 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2]
Layer spacings: [0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2

In [40]:
# 5/15 fixes to my code
'''From here to the next section is setting up the variables for the rest of the code'''
# Importing libraries that will be used for the rest of the code
import pyvista as pv
import csv
import pandas as pd
import numpy as np
from scipy.spatial import ConvexHull
from scipy.spatial.distance import pdist, squareform
from scipy.optimize import linear_sum_assignment
from shapely.geometry import Polygon
from shapely.ops import unary_union

# Load the STL and convert the points into mesh
mesh = pv.read('freeform_test_17v3.stl')
mesh = mesh.subdivide(3)  # Increase this number (e.g., 3 or 4) for more points
points = mesh.points

# Shift Z so the first point starts at Z = 50
z_shift = -50 - points[:, 2].min()
points[:, 2] += z_shift

pd.options.mode.copy_on_write = True

# Save mesh points to CSV so that the program can create the toolpath (this is all raw data)
with open('dots.csv', 'w') as f:
    csv.writer(f).writerows(points)
df = pd.read_csv('dots.csv', header=None, names=['x', 'y', 'z']).dropna()

# Remove the bottom layer of the shape in order to prevent the machine from going back over the already processed points
df = df[df['z'] > df['z'].min() + 0.1]

# Specify the desired layer spacing as a variable
layer_spacing = 0.2  # Set the distance between layers (in your desired units)

'''Modules for simplifying the tool path process'''

def enforce_layer_spacing(df, layer_spacing):
    """Ensure layers are spaced with a fixed distance defined by layer_spacing."""
    z_min = df['z'].min()
    layers = np.arange(z_min, df['z'].max() + layer_spacing, layer_spacing)
    df['z'] = layers[np.argmin(np.abs(df['z'].values[:, None] - layers), axis=1)]
    return df

df = enforce_layer_spacing(df, layer_spacing)

def is_circular_shape(points, tolerance=0.1):
    """Check if a set of points forms a roughly circular shape."""
    center = points.mean(axis=0)
    distances = np.linalg.norm(points - center, axis=1)
    return np.abs(distances - distances.mean()).max() < tolerance

# Module for calculating TSP (approximate)
def solve_tsp(points):
    distance_matrix = squareform(pdist(points))
    _, col_ind = linear_sum_assignment(distance_matrix)
    return col_ind

# Module for sorting the points in a clockwise direction
def sort_clockwise(points):
    center = points.mean(axis=0)
    angles = np.arctan2(points[:, 1] - center[1], points[:, 0] - center[0])
    return points[np.argsort(angles)]

# Module for processing circular layers
def process_circular_layer(layer_points):
    return sort_clockwise(layer_points)

# Module for processing non-circular layers
def process_non_circular_layer(layer_points):
    tsp_order = solve_tsp(layer_points)
    return sort_clockwise(layer_points[tsp_order])

# Reverse offset method
def offset_polygon(coords, offset):
    poly = Polygon(coords)
    if not poly.is_valid or poly.area == 0:
        return None
    offset_poly = poly.buffer(-offset)
    if offset_poly.is_empty:
        return None
    if offset_poly.geom_type == 'MultiPolygon':
        offset_poly = max(offset_poly.geoms, key=lambda p: p.area)
    return np.array(offset_poly.exterior.coords)

'''Creating the Tool path for the machine to process'''

# Initialize toolpath
ll = []
prev_exit_point = None
layer_index = 0
prev_layer_entry_area = None

while len(df) > 0:
    # This identifies the current layer by locating the Z with the largest absolute value
    current_layer_z = df.loc[df['z'].abs().idxmax(), 'z']
    current_layer = df[df['z'] == current_layer_z]

    # Skip layers with fewer than 3 points (ConvexHull requires at least 3)
    if len(current_layer) < 3:
        df = df[df['z'] != current_layer_z]
        continue

    # Filter points to retain only boundary points using ConvexHull
    layer_points = current_layer[['x', 'y']].values
    hull = ConvexHull(layer_points)
    hull_coords = layer_points[hull.vertices]

    # Apply reverse offset to generate outermost contour only
    offset_coords = offset_polygon(hull_coords, offset=1.25)
    if offset_coords is None or len(offset_coords) < 3:
        df = df[df['z'] != current_layer_z]
        continue
    hull_coords = offset_coords

    # Determine shape type
    if is_circular_shape(hull_coords):
        sorted_points = process_circular_layer(hull_coords)
    else:
        sorted_points = process_non_circular_layer(hull_coords)

    # Adjust start point to be closest to previous layer's end point
    if prev_exit_point is not None:
        dists = np.linalg.norm(sorted_points - prev_exit_point[:2], axis=1)
        start_idx = np.argmin(dists)
        sorted_points = np.roll(sorted_points, -start_idx, axis=0)

    # Add Z-coordinate to all points in the layer
    sorted_layer = np.hstack([sorted_points, np.full((sorted_points.shape[0], 1), current_layer_z)])

    # Prepare the toolpath for this layer
    layer_path = list(sorted_layer)

    # Save the last real point as exit for next layer transition
    prev_exit_point = layer_path[-1]

    # Add return to the first point to close the loop
    layer_path.append(layer_path[0])
    ll.extend(layer_path)

    # Remove processed points from DataFrame
    df = df[df['z'] != current_layer_z]

    # Print layer spacings to verify consistency
    unique_z = np.sort(df['z'].unique())
    print("Layer spacings:", np.diff(unique_z))

    layer_index += 1

# Save toolpath to a CSV
ll = np.round(ll, 3)
pd.DataFrame(ll, columns=['x', 'y', 'z']).to_csv('toolpath.csv', index=False)

'''Below is converting the tool path into G-Code and adding parameters to the G-code so that the tool will actually move during machining'''

# Convert to G-code and save as a text file
df_ltp = pd.read_csv('toolpath.csv')
df_gchord = pd.DataFrame({
    'G': 'G01',
    'x': 'X' + df_ltp['x'].astype(str),
    'y': 'Y' + df_ltp['y'].astype(str),
    'z': 'Z' + df_ltp['z'].astype(str),
    'EOB': ';'
})

# Define the starting and ending G-code lines
first_x = round(df_ltp.iloc[0]['x'], 3)
first_y = round(df_ltp.iloc[0]['y'], 3)
safe_z = 50  # Set your safe Z height here
start_z = -50  # Drop 50 mm below origin

gcode_header = [
    "G90 ; (Absolute positioning)",
    "G21 ; (Set units to millimeters)",
    "G92 X0.0 Y0.0 Z0 ; (Set workplace origin)",
    "S500 (setting spindle speed)",
    "F1000 ; (Set feedrate)",
    f"G01 X0 Y0 Z{start_z} ; (Lower 50 mm from origin)",
    f"G01 X{first_x} Y{first_y} Z{df_ltp.iloc[0]['z']} ; (Move to start point)"
]

last_x = round(df_ltp.iloc[-1]['x'], 3)
last_y = round(df_ltp.iloc[-1]['y'], 3)

gcode_footer = [
    f"G01 X{last_x} Y{last_y} Z{safe_z} ; (Retract tool straight up from last point)",
    "M30 ; (End of program)"
]

# Write final G-code file
with open('Gchord.txt', 'w') as f:
    for line in gcode_header:
        f.write(line + '\n')
    df_gchord.to_csv(f, index=False, sep=" ", header=False, mode='a')
    for line in gcode_footer:
        f.write(line + '\n')


Layer spacings: [0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2
 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2
 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2
 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2
 0.2]
Layer spacings: [0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2
 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2
 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2
 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2]
Layer spacings: [0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2
 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2
 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2
 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2]
Layer spacings: [0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2

In [37]:
# 5/20 - Reverse Offset with Toolpath Generation
import pyvista as pv
import csv
import pandas as pd
import numpy as np
from scipy.spatial import ConvexHull
from scipy.spatial.distance import pdist, squareform
from scipy.optimize import linear_sum_assignment

# Load and prepare mesh
mesh = pv.read('freeform_test_17.stl')
mesh = mesh.subdivide(2)
points = mesh.points
z_shift = -50 - points[:, 2].min()
points[:, 2] += z_shift

pd.options.mode.copy_on_write = True

with open('dots.csv', 'w') as f:
    csv.writer(f).writerows(points)

df = pd.read_csv('dots.csv', header=None, names=['x', 'y', 'z']).dropna()
df = df[df['z'] > df['z'].min() + 0.1]
layer_spacing = 0.2

# Apply fixed Z-layer spacing
def enforce_layer_spacing(df, layer_spacing):
    z_min = df['z'].min()
    layers = np.arange(z_min, df['z'].max() + layer_spacing, layer_spacing)
    df['z'] = layers[np.argmin(np.abs(df['z'].values[:, None] - layers), axis=1)]
    return df

df = enforce_layer_spacing(df, layer_spacing)

# Shape type determination
def is_circular_shape(points, tolerance=0.1):
    center = points.mean(axis=0)
    distances = np.linalg.norm(points - center, axis=1)
    return np.abs(distances - distances.mean()).max() < tolerance

# TSP solver for non-circular paths
def solve_tsp(points):
    distance_matrix = squareform(pdist(points))
    _, col_ind = linear_sum_assignment(distance_matrix)
    return col_ind

# Clockwise sorting
def sort_clockwise(points):
    center = points.mean(axis=0)
    angles = np.arctan2(points[:, 1] - center[1], points[:, 0] - center[0])
    return points[np.argsort(angles)]

def process_circular_layer(layer_points):
    return sort_clockwise(layer_points)

def process_non_circular_layer(layer_points):
    tsp_order = solve_tsp(layer_points)
    return sort_clockwise(layer_points[tsp_order])

# Apply reverse offset to a layer using dynamic wall angle

def apply_reverse_offset(prev_layer, current_layer, z_gap):
    if len(prev_layer) < 3 or len(current_layer) < 3:
        return current_layer  # Not enough points to compute offset

    # Match number of points
    prev = prev_layer[['x', 'y']].values
    curr = current_layer[['x', 'y']].values
    if len(curr) != len(prev):
        min_len = min(len(curr), len(prev))
        curr = curr[:min_len]
        prev = prev[:min_len]

    offset_points = []
    for (p_curr, p_prev) in zip(curr, prev):
        dz = z_gap
        dx, dy = p_curr[0] - p_prev[0], p_curr[1] - p_prev[1]
        horizontal_distance = np.sqrt(dx**2 + dy**2)
        angle_rad = np.arctan2(horizontal_distance, dz)

        # Calculate unit normal vector pointing outward
        if horizontal_distance == 0:
            offset_vector = np.array([0.0, 0.0])
        else:
            unit_dir = np.array([dx, dy]) / horizontal_distance
            offset_distance = dz * np.tan(angle_rad)
            offset_vector = unit_dir * offset_distance

        reverse_offset = p_curr - offset_vector
        offset_points.append(reverse_offset)

    offset_points = np.array(offset_points)
    offset_df = pd.DataFrame(offset_points, columns=['x', 'y'])
    offset_df['z'] = current_layer['z'].iloc[0]
    return offset_df

# Create the toolpath using reverse offset and maintain original logic
ll = []
prev_exit_point = None
prev_layer_df = None

while len(df) > 0:
    current_layer_z = df['z'].max()
    current_layer_df = df[df['z'] == current_layer_z]

    if len(current_layer_df) < 3:
        df = df[df['z'] < current_layer_z]
        continue

    layer_points = current_layer_df[['x', 'y']].values
    hull = ConvexHull(layer_points)
    hull_coords = layer_points[hull.vertices]
    hull_df = current_layer_df.iloc[hull.vertices]

    # Apply reverse offset if previous layer exists
    if prev_layer_df is not None:
        z_gap = current_layer_z - prev_layer_df['z'].iloc[0]
        hull_df = apply_reverse_offset(prev_layer_df, hull_df, z_gap)

    hull_points = hull_df[['x', 'y']].values
    if is_circular_shape(hull_points):
        sorted_points = process_circular_layer(hull_points)
    else:
        sorted_points = process_non_circular_layer(hull_points)

    if prev_exit_point is not None:
        dists = np.linalg.norm(sorted_points - prev_exit_point[:2], axis=1)
        start_idx = np.argmin(dists)
        sorted_points = np.roll(sorted_points, -start_idx, axis=0)

    sorted_layer = np.hstack([sorted_points, np.full((sorted_points.shape[0], 1), current_layer_z)])
    layer_path = list(sorted_layer)
    layer_path.append(layer_path[0])
    ll.extend(layer_path)
    prev_exit_point = layer_path[0]

    prev_layer_df = hull_df
    df = df[df['z'] < current_layer_z]
    unique_z = np.sort(df['z'].unique())
    print("Layer spacings:", np.diff(unique_z))

# Save toolpath
ll = np.round(ll, 3)
pd.DataFrame(ll, columns=['x', 'y', 'z']).to_csv('toolpath.csv', index=False)

# Generate G-code

df_ltp = pd.read_csv('toolpath.csv')
df_gchord = pd.DataFrame({
    'G': 'G01',
    'x': 'X' + df_ltp['x'].astype(str),
    'y': 'Y' + df_ltp['y'].astype(str),
    'z': 'Z' + df_ltp['z'].astype(str),
    'EOB': ';'
})

first_x = round(df_ltp.iloc[0]['x'], 3)
first_y = round(df_ltp.iloc[0]['y'], 3)
safe_z = 50
start_z = -50

gcode_header = [
    "G90 ; (Absolute positioning)",
    "G21 ; (Set units to millimeters)",
    "G92 X0.0 Y0.0 Z0 ; (Set workplace origin)",
    "S500 (setting spindle speed)",
    "F1000 ; (Set feedrate)",
    f"G01 X0 Y0 Z{start_z} ; (Lower 50 mm from origin)",
    f"G01 X{first_x} Y{first_y} Z{df_ltp.iloc[0]['z']} ; (Move to start point)"
]

last_x = round(df_ltp.iloc[-1]['x'], 3)
last_y = round(df_ltp.iloc[-1]['y'], 3)

gcode_footer = [
    f"G01 X{last_x} Y{last_y} Z{safe_z} ; (Retract tool straight up from last point)",
    "M30 ; (End of program)"
]

with open('Gchord.txt', 'w') as f:
    for line in gcode_header:
        f.write(line + '\n')
    df_gchord.to_csv(f, index=False, sep=" ", header=False, mode='a')
    for line in gcode_footer:
        f.write(line + '\n')


Layer spacings: [0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2
 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2
 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2
 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2
 0.2]
Layer spacings: [0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2
 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2
 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2
 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2]
Layer spacings: [0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2
 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2
 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2
 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2]
Layer spacings: [0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2

In [41]:
# 5/20 - Integrated ConvexHull-based toolpath generation
# Import required libraries
import pyvista as pv
import csv
import pandas as pd
import numpy as np
from scipy.spatial import ConvexHull
from scipy.spatial.distance import pdist, squareform
from scipy.optimize import linear_sum_assignment

# Load the STL file and prepare mesh points
mesh = pv.read('freeform_test_17v3.stl')
mesh = mesh.subdivide(2)  # Increase this for finer resolution
points = mesh.points

# Shift the mesh in Z so the lowest point is at Z = 50
z_shift = -50 - points[:, 2].max()
points[:, 2] += z_shift

# Save points to CSV and load into pandas DataFrame
pd.options.mode.copy_on_write = True
with open('dots.csv', 'w') as f:
    csv.writer(f).writerows(points)
df = pd.read_csv('dots.csv', header=None, names=['x', 'y', 'z']).dropna()

# Remove the very bottom layer to avoid reprocessing already-formed material
df = df[df['z'] > df['z'].min() + 0.1]

# Define desired vertical step between layers
layer_spacing = 0.2  # mm

# Function to snap all points to evenly spaced Z layers
def enforce_layer_spacing(df, layer_spacing):
    z_min = df['z'].min()
    layers = np.arange(z_min, df['z'].max() + layer_spacing, layer_spacing)
    df['z'] = layers[np.argmin(np.abs(df['z'].values[:, None] - layers), axis=1)]
    return df

df = enforce_layer_spacing(df, layer_spacing)

# Check if a group of 2D points roughly forms a circular shape
def is_circular_shape(points, tolerance=0.1):
    center = points.mean(axis=0)
    distances = np.linalg.norm(points - center, axis=1)
    return np.abs(distances - distances.mean()).max() < tolerance

# Solve TSP approximately using the linear sum assignment (Hungarian) method
def solve_tsp(points):
    distance_matrix = squareform(pdist(points))
    _, col_ind = linear_sum_assignment(distance_matrix)
    return col_ind

# Sort points in clockwise order around their centroid
def sort_clockwise(points):
    center = points.mean(axis=0)
    angles = np.arctan2(points[:, 1] - center[1], points[:, 0] - center[0])
    return points[np.argsort(angles)]

# For circular shapes, just sort clockwise
def process_circular_layer(layer_points):
    return sort_clockwise(layer_points)

# For non-circular shapes, solve approximate TSP then sort clockwise
def process_non_circular_layer(layer_points):
    tsp_order = solve_tsp(layer_points)
    return sort_clockwise(layer_points[tsp_order])

# Main toolpath generation loop
ll = []
prev_exit_point = None

while len(df) > 0:
    current_layer_z = df['z'].max()
    current_layer = df[df['z'] == current_layer_z]

    if len(current_layer) < 3:
        df = df[df['z'] < current_layer_z]
        continue

    layer_points_2d = current_layer[['x', 'y']].values

    # Attempt to extract outer boundary using ConvexHull
    try:
        hull = ConvexHull(layer_points_2d)
        hull_coords = layer_points_2d[hull.vertices]
    except Exception as e:
        print(f"Skipping layer at Z={current_layer_z:.3f} due to error: {str(e)}")
        df = df[df['z'] < current_layer_z]
        continue

    # Skip degenerate hulls (e.g., colinear points)
    if hull_coords.shape[0] < 3 or np.linalg.matrix_rank(hull_coords - hull_coords[0]) < 2:
        print(f"Skipping layer at Z={current_layer_z:.6f} due to colinear or degenerate points")
        df = df[df['z'] < current_layer_z]
        continue

    # Decide circular or non-circular path and sort
    if is_circular_shape(hull_coords):
        sorted_points = process_circular_layer(hull_coords)
    else:
        sorted_points = process_non_circular_layer(hull_coords)

    # Roll so path starts from closest point to previous layer
    if prev_exit_point is not None:
        dists = np.linalg.norm(sorted_points - prev_exit_point[:2], axis=1)
        start_idx = np.argmin(dists)
        sorted_points = np.roll(sorted_points, -start_idx, axis=0)

    # Add Z-coordinate to each point
    sorted_layer = np.hstack([sorted_points, np.full((sorted_points.shape[0], 1), current_layer_z)])

    # Return to first point for a closed path
    layer_path = list(sorted_layer)
    layer_path.append(layer_path[0])
    ll.extend(layer_path)

    # Store exit point for next layer
    prev_exit_point = layer_path[0]

    # Remove processed layer from DataFrame
    df = df[df['z'] < current_layer_z]

    # Print current Z-layer spacing
    unique_z = np.sort(df['z'].unique())
    print("Layer spacings:", np.diff(unique_z))

# Guard: Warn if no points were processed
if len(ll) == 0:
    print("Warning: No toolpath points generated. Check input data or layer filtering.")

# Save toolpath as CSV
ll = np.round(ll, 3)
pd.DataFrame(ll, columns=['x', 'y', 'z']).to_csv('toolpath.csv', index=False)

'''Below is converting the tool path into G-Code and adding parameters so that the tool will move during machining'''

# Load the path and convert to G-code format
df_ltp = pd.read_csv('toolpath.csv')
df_gchord = pd.DataFrame({
    'G': 'G01',
    'x': 'X' + df_ltp['x'].astype(str),
    'y': 'Y' + df_ltp['y'].astype(str),
    'z': 'Z' + df_ltp['z'].astype(str),
    'EOB': ';'
})

# Define safe heights and starting point
first_x = round(df_ltp.iloc[0]['x'], 3)
first_y = round(df_ltp.iloc[0]['y'], 3)
safe_z = 50
start_z = -50

gcode_header = [
    "G90 ; (Absolute positioning)",
    "G21 ; (Set units to millimeters)",
    "G92 X0.0 Y0.0 Z0 ; (Set workplace origin)",
    "S500 (setting spindle speed)",
    "F1000 ; (Set feedrate)",
    f"G01 X0 Y0 Z{start_z} ; (Lower 50 mm from origin)",
    f"G01 X{first_x} Y{first_y} Z{df_ltp.iloc[0]['z']} ; (Move to start point)"
]

last_x = round(df_ltp.iloc[-1]['x'], 3)
last_y = round(df_ltp.iloc[-1]['y'], 3)

gcode_footer = [
    f"G01 X{last_x} Y{last_y} Z{safe_z} ; (Retract tool straight up from last point)",
    "M30 ; (End of program)"
]

# Output the G-code
with open('Gchord.txt', 'w') as f:
    for line in gcode_header:
        f.write(line + '\n')
    df_gchord.to_csv(f, index=False, sep=" ", header=False, mode='a')
    for line in gcode_footer:
        f.write(line + '\n')


Layer spacings: [0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2
 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2
 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2
 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2
 0.2]
Layer spacings: [0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2
 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2
 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2
 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2]
Layer spacings: [0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2
 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2
 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2
 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2]
Layer spacings: [0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2

In [46]:
# 5/20 - Integrated ConvexHull-based toolpath generation with downscaling (reverse offset)

import pyvista as pv
import csv
import pandas as pd
import numpy as np
from scipy.spatial import ConvexHull
from scipy.spatial.distance import pdist, squareform
from scipy.optimize import linear_sum_assignment

# Load the STL file and prepare mesh points
mesh = pv.read('freeform_test_17v3.stl')
mesh = mesh.subdivide(2)  # Increase for finer resolution
points = mesh.points

# Shift mesh in Z so lowest point is at Z = 50
z_shift = -50 - points[:, 2].max()
points[:, 2] += z_shift

# Save points to CSV and load into DataFrame
pd.options.mode.copy_on_write = True
with open('dots.csv', 'w') as f:
    csv.writer(f).writerows(points)
df = pd.read_csv('dots.csv', header=None, names=['x', 'y', 'z']).dropna()

# Remove the very bottom layer
df = df[df['z'] > df['z'].min() + 0.1]

# Set vertical layer spacing
layer_spacing = 0.2  # mm

# Snap Z-coordinates to fixed layers
def enforce_layer_spacing(df, layer_spacing):
    z_min = df['z'].min()
    layers = np.arange(z_min, df['z'].max() + layer_spacing, layer_spacing)
    df['z'] = layers[np.argmin(np.abs(df['z'].values[:, None] - layers), axis=1)]
    return df

df = enforce_layer_spacing(df, layer_spacing)

# Check for circular shape
def is_circular_shape(points, tolerance=0.1):
    center = points.mean(axis=0)
    distances = np.linalg.norm(points - center, axis=1)
    return np.abs(distances - distances.mean()).max() < tolerance

# Approximate TSP
def solve_tsp(points):
    distance_matrix = squareform(pdist(points))
    _, col_ind = linear_sum_assignment(distance_matrix)
    return col_ind

# Sort points clockwise
def sort_clockwise(points):
    center = points.mean(axis=0)
    angles = np.arctan2(points[:, 1] - center[1], points[:, 0] - center[0])
    return points[np.argsort(angles)]

# Process circular layers
def process_circular_layer(layer_points):
    return sort_clockwise(layer_points)

# Process non-circular layers
def process_non_circular_layer(layer_points):
    tsp_order = solve_tsp(layer_points)
    return sort_clockwise(layer_points[tsp_order])

# Apply reverse offset to shrink layer boundary
def reverse_offset(points, scale_factor=2):
    center = points.mean(axis=0)
    return center + scale_factor * (points - center)

# Main toolpath generation loop
ll = []
prev_exit_point = None

while len(df) > 0:
    current_layer_z = df['z'].max()
    current_layer = df[df['z'] == current_layer_z]

    if len(current_layer) < 3:
        df = df[df['z'] < current_layer_z]
        continue

    layer_points_2d = current_layer[['x', 'y']].values

    try:
        hull = ConvexHull(layer_points_2d)
        hull_coords = layer_points_2d[hull.vertices]
    except Exception as e:
        print(f"Skipping layer at Z={current_layer_z:.3f} due to error: {str(e)}")
        df = df[df['z'] < current_layer_z]
        continue

    if hull_coords.shape[0] < 3 or np.linalg.matrix_rank(hull_coords - hull_coords[0]) < 2:
        print(f"Skipping layer at Z={current_layer_z:.6f} due to colinear or degenerate points")
        df = df[df['z'] < current_layer_z]
        continue

    # Downscale the boundary using reverse offset method
    downscaled_coords = reverse_offset(hull_coords, scale_factor=2)

    if is_circular_shape(downscaled_coords):
        sorted_points = process_circular_layer(downscaled_coords)
    else:
        sorted_points = process_non_circular_layer(downscaled_coords)

    # Start from closest point to previous exit
    if prev_exit_point is not None:
        dists = np.linalg.norm(sorted_points - prev_exit_point[:2], axis=1)
        start_idx = np.argmin(dists)
        sorted_points = np.roll(sorted_points, -start_idx, axis=0)

    # Add Z-coordinate and close path
    sorted_layer = np.hstack([sorted_points, np.full((sorted_points.shape[0], 1), current_layer_z)])
    layer_path = list(sorted_layer)
    layer_path.append(layer_path[0])
    ll.extend(layer_path)
    prev_exit_point = layer_path[0]

    df = df[df['z'] < current_layer_z]

# Guard: check if toolpath was generated
if len(ll) == 0:
    print("Warning: No toolpath points generated. Check input data or layer filtering.")

# Save toolpath to CSV
ll = np.round(ll, 3)
pd.DataFrame(ll, columns=['x', 'y', 'z']).to_csv('toolpath.csv', index=False)

'''Convert to G-code'''

df_ltp = pd.read_csv('toolpath.csv')
df_gchord = pd.DataFrame({
    'G': 'G01',
    'x': 'X' + df_ltp['x'].astype(str),
    'y': 'Y' + df_ltp['y'].astype(str),
    'z': 'Z' + df_ltp['z'].astype(str),
    'EOB': ';'
})

first_x = round(df_ltp.iloc[0]['x'], 3)
first_y = round(df_ltp.iloc[0]['y'], 3)
safe_z = 50
start_z = -50

gcode_header = [
    "G90 ; (Absolute positioning)",
    "G21 ; (Set units to millimeters)",
    "G92 X0.0 Y0.0 Z0 ; (Set workplace origin)",
    "S500 (setting spindle speed)",
    "F1000 ; (Set feedrate)",
    f"G01 X0 Y0 Z{start_z} ; (Lower 50 mm from origin)",
    f"G01 X{first_x} Y{first_y} Z{df_ltp.iloc[0]['z']} ; (Move to start point)"
]

last_x = round(df_ltp.iloc[-1]['x'], 3)
last_y = round(df_ltp.iloc[-1]['y'], 3)

gcode_footer = [
    f"G01 X{last_x} Y{last_y} Z{safe_z} ; (Retract tool straight up from last point)",
    "M30 ; (End of program)"
]

with open('Gchord.txt', 'w') as f:
    for line in gcode_header:
        f.write(line + '\n')
    df_gchord.to_csv(f, index=False, sep=" ", header=False, mode='a')
    for line in gcode_footer:
        f.write(line + '\n')
